# FieldData_Final_Pipeline

Clean replacement for `FieldData_Playground.ipynb` (117 cells, ~40% dev-history /
rejected experiments). This notebook implements **only** the methodology carried
forward into Chapter 7 ("Hypothesis 3: Generalisation to Real Field Data") of the
thesis, structured to mirror its five sections one-for-one:

- **7.1** Field Data Explanation
- **7.2** Processing and Migration (post-imaging processing now applies to
  Kirchhoff, Gazdag, *and* back-propagation — not just back-propagation)
- **7.3** Timelapse Differencing and Region of Influence Selection (napari
  picking only — no rectangular window, no sliding-window scan)
- **7.4** WLS Cross-Spectrum Phase Plane Fit (stage-anchored chaining only — no
  consecutive or fixed-baseline trajectory strategies)
- **7.5** Interpretation (WLS vs. RANSAC)

**Scope note.** Cells that need a live interactive session (napari picking) or an
external, long-running process (gprMax back-propagation FDTD runs) are marked
`# TODO (interactive)` / `# TODO (external gprMax run)` and contain the calling
code that would be used, not a working substitute — those steps genuinely need
your live environment to execute. Everything else in this notebook is real,
runnable Python, generalized/ported from `FieldData_Playground.ipynb` and
`helper_functions/`.

See `FieldData_Final_Pipeline_outline.md` alongside this notebook for the full
keep/discard/new function inventory this scaffold was built from.

## Setup

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from pylops.utils.wavelets import ricker

from gdp.data_io import load_mala
from gdp.preprocessing.filtering import filter_data, remove_mean
from gdp.preprocessing.gain import linear_gain
from gdp.preprocessing.image_processing import remove_svd
from gdp.preprocessing.trace_ops import align_traces

import helper_functions.KirchhoffPylopsZeroOffset as KirchhoffPylopsZeroOffset
from helper_functions.migration import (
    gazdag_migration,
    write_borehole_backprop_files,
    plot_borehole_domain,
    apply_3d_to_2d_correction,
    dispersion_limited_cutoff,
    load_migrated_image,
    wls_phase_plane_fit,
    ransac_phase_plane_fit,
    plot_phase_slice,
)
from helper_functions.figures import save_fig

# Raw data lives under fielddata/raw_data/060616/ as MALA-format prof / profN
# file pairs, read via gdp.data_io.load_mala -- NOT plain .npy files.
# helper_functions/processing.py duplicates some of the gdp.preprocessing.*
# functions imported above under the same names; FieldData_Playground.ipynb
# uses the gdp versions against the real raw data, so this notebook does too.
DATA = Path("fielddata") / "raw_data" / "060616"
OUT_DIR = Path("fielddata") / "output"
STUDY = "FieldData_Study"  # TimeLapse_Figures/<STUDY>/... curated tree, via save_fig

_prof_name = lambda n: "prof" if n == 0 else f"prof{n}"

# Survey parameters (Section 7.1)
V_PROP = 0.10        # m/ns, propagation velocity
F0_MIG = 0.10        # GHz, migration/centre frequency
DL = 0.05            # m, trace spacing
MAX_DEPTH = 85.0     # m
V_MIG = V_PROP / 2   # exploding-reflector convention (eq:vmig)
RAD_CUT = 300        # time samples kept for imaging

# Operational stages (Section 7.1)
STAGES = {
    "Push":  range(1, 5),
    "Chase": range(5, 10),
    "Wait":  range(10, 21),
    "Pull":  range(21, 39),
}

# The five profiles used throughout the chapter for cross-technique comparison,
# and the four representative consecutive pairs used for every displacement
# estimate (approximating Push/Chase/Wait/Pull with only four differenced pairs).
REPRESENTATIVE_PROFILES = [1, 3, 8, 20, 38]
STAGE_PAIRS = [(1, 3), (3, 8), (8, 20), (20, 38)]
STAGE_NAMES = ["Push", "Chase", "Wait", "Pull"]

## 7.1 Field Data Explanation

Load the 38 raw zero-offset profiles plus the pre-injection reference profile, and
produce the borehole-domain schematic used in `@fig:fd-borehole-schematic`. No
migration, differencing, or gain correction happens in this section — see 7.2.

In [ ]:
def load_raw_profile(run: int):
    """Load one raw MALA-format GPR profile (run=0 is the pre-injection
    reference profile prof.rd3/prof.rad). Returns (data, info); info carries
    the sampling frequency ('frequency (GHz)') among other acquisition
    metadata."""
    return load_mala(str(DATA / _prof_name(run)), return_object=False)


reference_raw, ref_info = load_raw_profile(0)
SF = ref_info["frequency (GHz)"]                                          # sampling frequency [GHz]
N_TRACES = reference_raw.shape[1]
N_SAMPLES = reference_raw.shape[0]
T_AXIS = np.arange(1, N_SAMPLES + 1) / SF                                 # two-way travel time [ns]
DEPTH_AXIS = np.linspace(MAX_DEPTH, MAX_DEPTH - N_TRACES * DL, N_TRACES)  # along-borehole depth [m]

# Run 6 is missing from the acquisition sequence (FieldData_Playground.ipynb
# cell [5]: `list(range(0, 6)) + list(range(7, 39))`) -- 37 profiles remain,
# matching the chapter's "all 37 processed profiles" figure.
PROFILE_RUNS = list(range(1, 6)) + list(range(7, 39))
raw_profiles = {run: load_raw_profile(run)[0] for run in PROFILE_RUNS}

In [ ]:

# Faithful port of FieldData_Playground.ipynb cell [12]: a self-contained
# demo of the borehole-geometry domain (representative 60-85 m depth range,
# placeholder trace data) used purely to build and sanity-check the domain
# and draw the schematic. No gprMax run needed for this cell -- the real,
# per-profile .in files (with real trace data) are written in 7.2.
BH_WIDTH, BH_EPS_R, BH_SIGMA = 0.10, 81.0, 0.01          # [m], water permittivity, water conductivity [S/m]
BH_LEFT_BUF, BH_IMG_RNG, BH_SRC_OFFSET = 1.0, 12.0, 3.2  # [m] clearance left / imaging range right / Tx-Rx offset
BP_DX = V_PROP / (F0_MIG * 20)                            # [m] grid spacing, lambda/20 at f0_mig
BP_PML = 15                                               # PML cells

z_bh_demo = np.arange(60.0, 85.0 + 1e-9, DL)
data_demo = np.zeros((len(z_bh_demo), RAD_CUT))  # placeholder trace data -- geometry only

_, _, _, _, geom_demo = write_borehole_backprop_files(
    OUT_DIR, label="borehole geometry demo", slug="borehole_demo",
    tapered_ntr_nt=data_demo, dt_ns=1.0, x_midpoints=z_bh_demo,
    t0_ns=200.0, eps_r=(0.299792458 / V_PROP) ** 2, v_ice=V_PROP,
    eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
    borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
    src_offset=BH_SRC_OFFSET, dx=BP_DX, pml_cells=BP_PML,
)

fig = plot_borehole_domain(geom_demo, x_src_true=z_bh_demo[::20])
save_fig(fig, "borehole_domain_schematic", study=STUDY, prefix="FD_", category="Compilations")

## 7.2 Processing and Migration

Shared five-step conditioning chain, then migration (Kirchhoff, Gazdag,
back-propagation), then **universal post-imaging processing** applied to all
three techniques' output — the central methodological change from the previous
notebook, where the fan/gate/taper recipe (`compute_final_processed`) was
hardcoded to back-propagation's gprMax focus frames only.

**Discarded from `FieldData_Playground.ipynb`:** the dev-history clutter-removal
experiments (mean/SVD/sliding-window/derivative variants, Mode 1/2/X12,
Gaussian+RPCA — cells [22]-[36], all self-identified rejected/negative results);
the windowed fan-filter variant (rejected — restricting the FFT to a local
window doesn't stop locally-oriented noise being reconstructed within it); the
homogeneous-domain (dx=0.05, no explicit borehole) back-propagation approach and
its `_load_bp_frame_wls` / `BP_FOCUS_IDX_OFFSET` loader (cells [4]-[8], [76]-[79]
— explicitly superseded in the old notebook, but still silently used by its
thesis-figure cells; not carried forward here at all).

In [ ]:
def preprocess_profile(raw, align_to, *, align_reference=False, normalize_align=True):
    """Shared 5-step conditioning chain -- bandpass, DC removal, trace
    alignment, SVD suppression, linear gain -- matching
    FieldData_Playground.ipynb's per-run processing (cells [5]/[8]), pulled
    into one reusable function. `align_to` is the reference's *own* aligned
    array (see reference_bscan below); time-lapse differencing against it
    happens separately in 7.3, not inside this function.

    Returns a B-scan of shape (n_traces, RAD_CUT), traces first, ready for
    migration."""
    n = min(raw.shape[1], N_TRACES)
    x = filter_data(raw[:, :n], fq=(0.02, 0.20), sfreq=SF, btype="bandpass")
    x, _ = remove_mean(x, 299, 517)
    x, _, _ = align_traces(x, align_to[:, :n], upsample=5,
                            normalize=normalize_align, align_reference=align_reference)
    x, _ = remove_svd(x, low_s=0, high_s=1)
    x, _ = linear_gain(x, T_AXIS)
    return x[:RAD_CUT, :].T


# Reference profile aligns to itself (align_reference=True); every other
# profile then aligns to *this* aligned reference array.
_ref_bp = filter_data(reference_raw, fq=(0.02, 0.20), sfreq=SF, btype="bandpass")
_ref_dc, _ = remove_mean(_ref_bp, 299, 517)
ref_aligned, _, _ = align_traces(_ref_dc, _ref_dc, upsample=5, normalize=False, align_reference=True)

reference_bscan = preprocess_profile(reference_raw, ref_aligned, align_reference=True, normalize_align=False)
conditioned = {run: preprocess_profile(raw, ref_aligned) for run, raw in raw_profiles.items()}

In [ ]:
def _ricker_wavelet(t_mig, f0=F0_MIG):
    period = 1.0 / f0
    n_wav = int(np.ceil(6 * period / (t_mig[1] - t_mig[0])))
    if n_wav % 2 == 0:
        n_wav += 1
    wav, _, wcenter = ricker(t_mig[:n_wav], f0=f0)
    return wav, wcenter


T_MIG = T_AXIS[:RAD_CUT]
X_IMG = np.linspace(0, V_PROP * T_MIG[-1] / 2, RAD_CUT)   # radial-distance image axis, shared by all profiles
DX_IMG = X_IMG[1] - X_IMG[0]                               # NOTE: not equal to DL -- separate axis/spacing
_WAV_RICKER, _WCENTER = _ricker_wavelet(T_MIG)
_WAV_DELTA = np.zeros_like(_WAV_RICKER)
_WAV_DELTA[_WCENTER] = 1.0   # "Kirchhoff-BP" variant: delta wavelet instead of Ricker


def migrate_kirchhoff(bscan, n_traces, delta_wavelet=False):
    """Kirchhoff migration via the operator directly
    (helper_functions/KirchhoffPylopsZeroOffset.py), matching
    FieldData_Playground.ipynb cell [8]'s construction rather than the
    PylopsKirchoffMigration wrapper -- the wrapper's documented (n_t, n_x)
    data-axis convention doesn't match this codebase's actual (n_traces,
    RAD_CUT) B-scan layout (dimsd=(n, nt) per the Kirchhoff operator itself),
    so calling it directly here avoids a silent transpose bug.
    delta_wavelet=True gives the "Kirchhoff-BP" variant used throughout the
    chapter (a delta wavelet instead of the Ricker). Expects `bscan` traces-
    first, (n_traces, RAD_CUT) -- exactly what conditioned[run] already is."""
    z_bh = DEPTH_AXIS[:n_traces]
    z_img = np.linspace(z_bh.max(), z_bh.min(), n_traces)
    recs = np.vstack((np.zeros(n_traces), z_bh))
    wav = _WAV_DELTA if delta_wavelet else _WAV_RICKER
    K = KirchhoffPylopsZeroOffset.Kirchhoff(z=z_img, x=X_IMG, t=T_MIG, srcs=recs, recs=recs,
                                             vel=V_PROP, wav=wav, wavcenter=_WCENTER,
                                             mode="analytic", dynamic=False)
    m = (K.H @ bscan.flatten()).reshape(len(X_IMG), len(z_img)).T
    return m  # (n_depth, n_radial)


def migrate_gazdag(bscan, n_traces):
    """Gazdag phase-shift migration (helper_functions/migration.py).

    Unlike Kirchhoff, gazdag_migration()'s own docstring requires `data`
    shaped (n_t, n_x) -- time-first, the OPPOSITE of conditioned[run]'s
    (n_traces, RAD_CUT) traces-first layout (confirmed against
    FieldData_Playground.ipynb cell [5], which passes `ref_gain[:rad_cut,:]`
    un-transposed to gazdag_migration, but its transposed twin `Dt_ref` to
    Kirchhoff). Feeding it the traces-first array directly (as an earlier
    version of this function did) silently migrated along the wrong axis --
    Kirchhoff-BP was unaffected since it genuinely wants traces-first, which
    is why only Gazdag's output looked wrong. `bscan.T` below is the fix."""
    z_bh = DEPTH_AXIS[:n_traces]
    x_bh = np.arange(n_traces) * DL
    return gazdag_migration(bscan.T, x_bh, T_MIG, X_IMG, V_PROP).T  # (n_depth, n_radial)


# Reference migrations, subtracted from every other profile's migration below
# (run 0's own migration is zero by self-subtraction, so profile 0 itself is
# never a key in the migrated_* dicts -- only used here as the subtrahend).
ref_kirchhoff = migrate_kirchhoff(reference_bscan, N_TRACES, delta_wavelet=False)
ref_kirchhoff_bp = migrate_kirchhoff(reference_bscan, N_TRACES, delta_wavelet=True)
ref_gazdag = migrate_gazdag(reference_bscan, N_TRACES)

# Kirchhoff, Kirchhoff-BP and Gazdag: linear (1/r) gain correction already
# folded into preprocess_profile's linear_gain step above, then migrate and
# subtract the reference. Available for all 37 processed profiles.
migrated_kirchhoff, migrated_kirchhoff_bp, migrated_gazdag = {}, {}, {}
for run, bscan in conditioned.items():
    n = bscan.shape[0]
    migrated_kirchhoff[run] = migrate_kirchhoff(bscan, n, delta_wavelet=False) - ref_kirchhoff[:n, :]
    migrated_kirchhoff_bp[run] = migrate_kirchhoff(bscan, n, delta_wavelet=True) - ref_kirchhoff_bp[:n, :]
    migrated_gazdag[run] = migrate_gazdag(bscan, n) - ref_gazdag[:n, :]

In [ ]:
# TODO (external gprMax run): write per-profile .in files for the sole
# back-propagation configuration used in this chapter (dx=0.02 m, explicit
# borehole geometry), for the five representative profiles. Faithful port of
# FieldData_Playground.ipynb cells [13]/[19] (differencing -> 3D-to-2D
# correction -> Tukey tapers -> evanescent filter -> write), plus the
# cross-profile registration fix of chapter 7.2.1: a single amplitude shared
# across all five profiles' excitation data (norm_scale=), replacing
# write_borehole_backprop_files' default per-trace peak normalisation.
# Discarded: write_backprop_files (dx=0.05, homogeneous domain, no borehole
# material) -- the only back-propagation configuration used in this thesis now.
from scipy.signal.windows import tukey as _tukey

DX02 = 0.02
_EPS_R = (0.299792458 / V_PROP) ** 2
_T0_NS = 299.0 / SF
_EPS_R_HALF = 4.0 * _EPS_R
_F_CUT_HZ = dispersion_limited_cutoff(_EPS_R_HALF, DX02)


def _tapered_excitation(run):
    """Reproduces the differenced, 3D-to-2D-corrected, Tukey-tapered,
    evanescent-filtered excitation data write_borehole_backprop_files expects
    (FieldData_Playground.ipynb cells [13]/[19])."""
    n = conditioned[run].shape[0]
    diff = conditioned[run][:, :RAD_CUT] - reference_bscan[:n, :RAD_CUT]
    diff_2d = apply_3d_to_2d_correction(diff, dt=(1.0 / SF) * 1e-9, velocity=V_PROP * 1e9, time_zero_idx=299)
    tapered = diff_2d * _tukey(n, alpha=0.30)[:, np.newaxis]
    tapered = tapered * _tukey(tapered.shape[1], alpha=0.10)[np.newaxis, :]
    kz = np.fft.fftfreq(tapered.shape[0], d=DL)
    freq = np.fft.rfftfreq(tapered.shape[1], d=1.0 / SF)
    D_fk = np.fft.fft(np.fft.rfft(tapered, axis=1), axis=0)
    D_fk[np.abs(kz[:, None]) > np.abs(freq[None, :]) / (V_PROP / 2)] = 0.0
    return np.real(np.fft.irfft(np.fft.ifft(D_fk, axis=0), n=tapered.shape[1], axis=1))


tapered_excitations = {run: _tapered_excitation(run) for run in REPRESENTATIVE_PROFILES}

# Cross-profile registration fix (7.2.1): one amplitude shared across all
# five profiles' excitation data, not a per-trace peak.
SHARED_NORM_SCALE = max(np.max(np.abs(v)) for v in tapered_excitations.values())

borehole_in_paths = {}
for run in REPRESENTATIVE_PROFILES:
    n = conditioned[run].shape[0]
    slug = f"borehole_prof_{run}_dx02"
    in_path, n_src, n_snaps, t_focus_ns, geom = write_borehole_backprop_files(
        OUT_DIR, label=slug, slug=slug,
        tapered_ntr_nt=tapered_excitations[run], dt_ns=1.0 / SF, x_midpoints=DEPTH_AXIS[:n],
        t0_ns=_T0_NS, eps_r=_EPS_R, v_ice=V_PROP,
        eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
        borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
        src_offset=BH_SRC_OFFSET, dx=DX02, pml_cells=BP_PML,
        normalize_mode="peak", norm_scale=SHARED_NORM_SCALE,
    )
    borehole_in_paths[run] = in_path

print(f"wrote {len(borehole_in_paths)} .in files -- run these through gprMax externally, "
      f"then load_backprop_focus_frame() below will find real snapshot data for them.")

# ---------------------------------------------------------------------------
# Snapshot-reading: real, faithful port of FieldData_Playground.ipynb's
# _best_borehole_focus_variant (cell [17]), the CORRECTED version that
# replaced an independent per-profile peak-amplitude snapshot search (which
# let different profiles lock onto different focus *times*, smearing the
# time-lapse difference) with the fixed-offset approach chapter 7.2.1
# describes. Needs real gprMax .vti snapshot output on disk to run -- until
# you've actually run gprMax on the .in files above, it raises a clear
# FileNotFoundError rather than a confusing downstream crash.
import pyvista

BH_NEAR_MASK_M = 1.0             # [m] past the borehole's outer wall to mask (injection halo)
N_SNAP_BH, SNAP_WIN_BH = 30, 1.0  # must match write_borehole_backprop_files' defaults (n_snap, snap_win)
_dt_ns_bh = 1.0 / SF
_T_ns_bh = RAD_CUT * _dt_ns_bh
_t_focus_ns_bh = _T_ns_bh - _T0_NS
_t_start_ns_bh = max(_dt_ns_bh, _t_focus_ns_bh - SNAP_WIN_BH)
_snap_step_bh = max(1, int((_T_ns_bh - _t_start_ns_bh) / (max(1, N_SNAP_BH - 1) * _dt_ns_bh)))


def load_backprop_focus_frame(run, focus_idx_offset):
    """Read back the dx=0.02 borehole-geometry gprMax snapshots for `run`,
    apply the near-borehole injection-halo mask, and return the single
    focus-time frame selected by the fixed snapshot-index offset (calibrated
    once against profile 1, chapter 7.2.1) -- NOT an independent per-profile
    peak-amplitude search, which was the pre-fix behaviour that broke
    cross-profile registration. Returns (ez_masked, depth_axis, radial_axis):
    the masked Ez frame as a (depth, radial) ndarray, plus its true-metre
    depth and radial-distance axes -- back-propagation's native grid (0.02 m,
    origin set by the borehole geometry) doesn't line up with Kirchhoff/
    Gazdag's (DEPTH_AXIS/X_IMG), so callers need these to plot it correctly.

    Raises FileNotFoundError if no .vti snapshots exist yet for this run
    (i.e. the .in file above hasn't been run through gprMax)."""
    slug = f"borehole_prof_{run}_dx02"
    snap_dir = OUT_DIR / "backprop" / slug / f"backprop_{slug}_snaps"
    snap_files = sorted(snap_dir.glob("bp_snap*.vti"), key=lambda p: int(p.stem.replace("bp_snap", "")))
    if not snap_files:
        raise FileNotFoundError(
            f"No gprMax snapshots found for {slug} under {snap_dir}. "
            f"Run gprMax on {borehole_in_paths.get(run, '<.in file above>')} first.")

    n = conditioned[run].shape[0]
    pml_pad = BP_PML * DX02
    depth_buffer = BH_SRC_OFFSET / 2.0
    x_shift = pml_pad + depth_buffer - float(np.min(DEPTH_AXIS[:n]))
    y_bh_end = pml_pad + BH_LEFT_BUF + BH_WIDTH

    snap_times_ns = _t_start_ns_bh + np.arange(len(snap_files)) * _snap_step_bh * _dt_ns_bh
    nearest_idx = int(np.argmin(np.abs(snap_times_ns - _t_focus_ns_bh)))
    k = min(nearest_idx + focus_idx_offset, len(snap_files) - 1)

    mesh = pyvista.read(str(snap_files[k]))
    nx_c, ny_c = mesh.dimensions[0] - 1, mesh.dimensions[1] - 1
    dx_m = float(mesh.spacing[0])
    ez = np.array(mesh["E-field"])[:, 2].reshape(ny_c, nx_c).T  # (depth, radial)
    mask_px = max(1, round((y_bh_end + BH_NEAR_MASK_M) / dx_m))
    ez_masked = ez.copy()
    ez_masked[:, :mask_px] = 0.0

    depth_axis = np.arange(nx_c) * dx_m - x_shift
    radial_axis = np.arange(ny_c) * dx_m - y_bh_end
    return ez_masked, depth_axis, radial_axis


FOCUS_IDX_OFFSET = 25  # calibrated once against profile 1 (see chapter 7.2.1)

In [ ]:
def apply_post_imaging(image, dz, dx, technique, *,
                        amp_gate_sigma_m=0.15, amp_gate_percentile=80,
                        radial_taper=None):
    """Technique-agnostic post-imaging clean-up: local-envelope amplitude
    gate, plus an optional near-borehole radial taper for back-propagation.

    The f-k dip (fan) filter originally in this function (ported from
    FieldData_Playground.ipynb's compute_final_processed(), cell [44]) has
    been dropped: tested against the real Gazdag and back-propagation
    images, it flattened genuine reflector energy rather than cleaning up
    noise, making both look worse than the amplitude-gated image alone.
    _estimate_dip/fan_taper_mask are kept below, unused, in case a
    per-technique fan filter is worth revisiting later.

    Parameters
    ----------
    image : ndarray (n_z, n_x)
        A single migrated image (already migrated, not yet post-processed).
    dz, dx : float
        Depth / radial pixel spacing (m). Both now genuinely used: they
        convert amp_gate_sigma_m into a per-axis PIXEL sigma, and dx also
        converts the radial taper distances to pixels.
    technique : {'kirchhoff_bp', 'gazdag', 'backprop'}
        Only used to decide whether the near-borehole radial taper applies
        (back-propagation only -- it exists to suppress the injection halo,
        which Kirchhoff/Gazdag don't have).
    amp_gate_sigma_m : float
        Amplitude-gate envelope smoothing, in METRES -- not a pixel count.
        Converted internally to (amp_gate_sigma_m/dz, amp_gate_sigma_m/dx),
        so every technique gets the same PHYSICAL smoothing distance despite
        very different native pixel spacing: Kirchhoff-BP/Gazdag image onto
        the real survey's own trace/time sampling (dz=DL~5.0 cm,
        dx=DX_IMG~4.4 cm), while back-propagation images onto gprMax's
        dedicated FDTD mesh (dz=dx=DX02=2.0 cm, ~2.5x finer). A fixed PIXEL
        sigma (the previous amp_gate_sigma=3 default) therefore smoothed
        Kirchhoff-BP/Gazdag over ~2.5x more physical distance than
        back-propagation from the exact same line of code -- an unintended,
        uneven blur on top of the real grid-density difference between the
        techniques, which is why back-propagation's B-scans looked sharper/
        higher-resolution even where the underlying signal quality was
        comparable. Default 0.15 m reproduces the old 3-pixel behaviour on
        Kirchhoff-BP/Gazdag's grid unchanged; back-propagation now gets the
        equivalent ~7.5px smoothing on its finer grid instead of 3px.
    radial_taper : (start_m, end_m) or None
        Cosine-ramped near-borehole taper. Defaults to (0, 3.0) for
        technique == 'backprop', None otherwise.
    """
    if radial_taper is None and technique == "backprop":
        radial_taper = (0.0, 3.0)

    sigma_px = (amp_gate_sigma_m / dz, amp_gate_sigma_m / dx)
    envelope = gaussian_filter(np.abs(image), sigma_px)
    gate = np.clip(envelope / np.percentile(envelope, amp_gate_percentile), 0, 1)
    gated = image * gate

    if radial_taper is not None:
        start_px, end_px = int(radial_taper[0] / dx), int(radial_taper[1] / dx)
        ramp = np.clip((np.arange(image.shape[1]) - start_px) / max(end_px - start_px, 1), 0, 1)
        gated = gated * ramp[np.newaxis, :]

    return gated


def _estimate_dip(image):
    """Per-column peak-pick + linear fit -- ported from cells [37]-[38].
    Not called by apply_post_imaging any more (see its docstring)."""
    peak_rows = np.argmax(np.abs(image), axis=0)
    cols = np.arange(image.shape[1])
    slope, _ = np.polyfit(cols, peak_rows, 1)
    return slope


def fan_taper_mask(shape, dip, hw_deg, taper_deg):
    """Cosine-tapered wedge around `dip` in the (kz, kx) domain -- ported
    unchanged from FieldData_Playground.ipynb (cells [37]-[42]). Not called
    by apply_post_imaging any more (see its docstring): it made Gazdag and
    back-propagation post-imaging results look worse, not cleaner."""
    n_z, n_x = shape
    kz = np.fft.fftfreq(n_z)[:, None]
    kx = np.fft.fftfreq(n_x)[None, :]
    theta = np.degrees(np.arctan2(kz, kx)) - np.degrees(np.arctan(dip))
    theta = (theta + 90) % 180 - 90
    ramp = np.clip((hw_deg + taper_deg - np.abs(theta)) / taper_deg, 0, 1)
    return ramp

In [ ]:
# Apply post-imaging uniformly to all three techniques, for the five
# representative profiles -- this is the fix for the pipeline inconsistency
# found in the old notebook (post-imaging was back-propagation-only, and even
# the thesis-figure cells bypassed it via the superseded loader).
# NOTE: dz=DL (borehole depth-axis spacing) but dx=DX_IMG, not DL -- the
# migrated image's radial axis (X_IMG) has its own spacing, unrelated to the
# along-borehole trace spacing.
post_imaged = {"kirchhoff_bp": {}, "gazdag": {}, "backprop": {}}
backprop_axes = {}  # run -> (depth_axis, radial_axis), back-propagation's own true-metre grid
for run in REPRESENTATIVE_PROFILES:
    post_imaged["kirchhoff_bp"][run] = apply_post_imaging(
        migrated_kirchhoff_bp[run], dz=DL, dx=DX_IMG, technique="kirchhoff_bp")
    post_imaged["gazdag"][run] = apply_post_imaging(
        migrated_gazdag[run], dz=DL, dx=DX_IMG, technique="gazdag")
    try:
        ez, depth_axis, radial_axis = load_backprop_focus_frame(run, FOCUS_IDX_OFFSET)
        post_imaged["backprop"][run] = apply_post_imaging(ez, dz=DX02, dx=DX02, technique="backprop")
        backprop_axes[run] = (depth_axis, radial_axis)
    except FileNotFoundError as e:
        # gprMax hasn't been run for this profile yet -- Kirchhoff-BP/Gazdag
        # still populate normally; back-propagation fills in once you have.
        print(f"[skip] profile {run}: {e}")

In [ ]:
import matplotlib.ticker as mticker


def build_profile_migration_grid(profiles, processed_bscans, post_imaged, backprop_axes,
                                   depth_range=(62, 86), sc=1.2):
    """Assemble Figure 7.2: 4 columns (processed B-scan / Kirchhoff-BP /
    Gazdag / back-propagation) x len(profiles) rows, all post-imaging
    processed. Styled to match FieldData_Playground.ipynb cell [92]'s
    profile_migration_grid.png: figure size, per-panel symmetric colour
    scaling (lim = max(sc * |image|.max(), 1.0)), seismic colormap, shared
    depth/radial ranges, 2 m / 5 m tick spacing, column titles, suptitle.

    Replaces the old notebook's cell [92] in one more way: its back-
    propagation column used the superseded loader (raw homogeneous-domain
    frame, no post-imaging); this one uses the current dx=0.02 borehole
    pipeline's post-imaging-processed frame instead, via post_imaged/
    backprop_axes from the cells above.

    Tolerant of a technique missing a given profile (e.g. back-propagation
    before its gprMax run has completed): draws a "pending" placeholder
    panel instead of raising."""
    col_titles = ["Processed B-scan", "Kirchhoff-BP migration", "Gazdag migration",
                  "Back-propagation ($E_z$ focus frame)"]
    radial_range = (0.0, float(X_IMG[-1]))

    fig, axes = plt.subplots(len(profiles), 4, figsize=(17, 3.6 * len(profiles)),
                              sharex=True, sharey=True)

    for row, run in enumerate(profiles):
        # Columns 0-2: processed B-scan, Kirchhoff-BP, Gazdag -- all share the
        # DEPTH_AXIS / X_IMG grid.
        for col, panel in enumerate((processed_bscans.get(run),
                                      post_imaged["kirchhoff_bp"].get(run),
                                      post_imaged["gazdag"].get(run))):
            ax = axes[row, col]
            if panel is None:
                ax.text(0.5, 0.5, "pending", ha="center", va="center", transform=ax.transAxes, fontsize=15)
                continue
            n = panel.shape[0]
            z = DEPTH_AXIS[:n]
            lim = max(sc * np.max(np.abs(panel)), 1.0)
            ax.imshow(panel, aspect="auto", cmap="seismic",
                      extent=[radial_range[0], radial_range[1], z[-1], z[0]],
                      vmin=-lim, vmax=lim, origin="upper")
            ax.invert_yaxis()
            ax.tick_params(axis='both', labelsize=20)

        # Column 3: back-propagation -- its own native grid (backprop_axes),
        # not DEPTH_AXIS/X_IMG.
        ax = axes[row, 3]
        panel = post_imaged["backprop"].get(run)
        if panel is None:
            ax.text(0.5, 0.5, "pending", ha="center", va="center", transform=ax.transAxes, fontsize=15)
        else:
            depth_axis, radial_axis = backprop_axes[run]
            lim = max(sc * np.max(np.abs(panel)), 1.0)
            ax.imshow(panel, aspect="auto", cmap="seismic",
                      extent=[radial_axis[0], radial_axis[-1], depth_axis[-1], depth_axis[0]],
                      vmin=-lim, vmax=lim, origin="upper")
            ax.invert_yaxis()

        for col in range(4):
            ax = axes[row, col]
            ax.set_ylim(depth_range[1], depth_range[0])
            ax.set_xlim(*radial_range)
            ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
            ax.yaxis.set_major_locator(mticker.MultipleLocator(5))
            if row == len(profiles) - 1:
                ax.set_xlabel("Radial distance (m)", fontsize=18)
            ax.tick_params(axis='both', labelsize=20)
        axes[row, 0].set_ylabel(f"Profile {run}\nDepth (m)", fontsize=18)
        
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=18)

    fig.suptitle("Processed profiles and their migrated / back-propagated counterparts", y=1.005, fontsize=20)
    fig.tight_layout()
    return fig


fig = build_profile_migration_grid(REPRESENTATIVE_PROFILES, conditioned, post_imaged, backprop_axes)
save_fig(fig, "profile_migration_grid", study=STUDY, prefix="FD_", category="Compilations")

## 7.3 Timelapse Differencing and Region of Influence Selection

Differencing between the four representative pairs, then ROI selection by napari
picking only.

**Discarded from `FieldData_Playground.ipynb`:** `sliding_window_scan` /
`_plot_window_qc` / `_plot_disp_maps` (cells [68]-[69], the sliding-window
scanner); the rectangular-window cell [67] and its `MANUAL_ROIS_BY_METHOD`
hand-picked-box fallback / `_roi_from_envelope` automatic-threshold fallback.

In [ ]:
def difference_pair(method, run_a, run_b, post_imaged):
    """Time-lapse difference for one representative pair: the later
    post-imaging-processed image minus the earlier one."""
    return post_imaged[method][run_b] - post_imaged[method][run_a]


differences = {
    method: {(a, b): difference_pair(method, a, b, post_imaged) for a, b in STAGE_PAIRS}
    for method in ("kirchhoff_bp", "gazdag", "backprop")
}

In [ ]:
# Per-technique fit constants + WLS defaults, shared by 7.3's picking cells
# below and 7.4/7.5's fitting cells (via _fit_inputs, defined at the start of
# 7.4).
#
# kz_cent must match load_migrated_image's convention (helper_functions/
# migration.py): Kirchhoff-BP/Gazdag are already migrated into the full-
# velocity domain, so kz_cent = 2*pi*f0/v (V_PROP). Back-propagation's raw Ez
# focus frame is still in the v/2 exploding-reflector domain gprMax itself
# operates in, so kz_cent = 2*pi*f0/(v/2) (V_MIG) -- using V_MIG for all three
# methods (as an earlier draft of this notebook did) silently halves
# Kirchhoff-BP/Gazdag's fitting band. Likewise dz/dx: Kirchhoff-BP/Gazdag live
# on the DEPTH_AXIS/X_IMG grid (spacing DL/DX_IMG); back-propagation lives on
# its own dx=0.02 borehole grid (DX02) via load_backprop_focus_frame, which is
# unrelated to DL/DX_IMG.
KZ_CENT_BY_METHOD = {
    "kirchhoff_bp": 2 * np.pi * F0_MIG / V_PROP,
    "gazdag": 2 * np.pi * F0_MIG / V_PROP,
    "backprop": 2 * np.pi * F0_MIG / V_MIG,
}
DZ_BY_METHOD = {"kirchhoff_bp": DL, "gazdag": DL, "backprop": DX02}
DX_BY_METHOD = {"kirchhoff_bp": DX_IMG, "gazdag": DX_IMG, "backprop": DX02}

# A fitted dz_est is only meaningful once you know which physical direction
# "increasing row index" points in the image that produced it -- and that
# direction is OPPOSITE for back-propagation versus Kirchhoff-BP/Gazdag:
#   - Kirchhoff/Gazdag images are indexed by DEPTH_AXIS/z_img, which DECREASES
#     with row index (row 0 = MAX_DEPTH = 85 m, deepest; row increases ->
#     shallower/upward).
#   - load_backprop_focus_frame's depth_axis = arange(nx_c)*dx_m - x_shift
#     INCREASES with row index (row 0 = shallowest; row increases -> deeper/
#     downward) -- the opposite convention, inherited from gprMax's own grid
#     indexing, never flipped to match.
# A positive raw dz_est therefore means "toward increasing row index", which
# is upward for Kirchhoff/Gazdag but downward for back-propagation -- the
# same-signed number means opposite physical directions. This is the same
# sign-convention issue load_migrated_image's docstring documents (and fixes
# architecturally, by flipping the image itself) for the old notebook; this
# notebook instead corrects the SIGN OF THE FITTED dz WHEREVER IT IS REPORTED
# OR CHAINED via _physical_dz below, leaving the raw fit (and its diagnostic
# plots, which must stay in the image's native row convention to actually
# overlay the data) untouched.
DZ_SIGN_BY_METHOD = {"kirchhoff_bp": 1, "gazdag": 1, "backprop": -1}


def _physical_dz(method, dz_est):
    """Sign-correct a raw wls_phase_plane_fit/ransac_phase_plane_fit dz_est
    into a consistent physical direction across all three techniques -- call
    this on every dz_est that gets reported, tabulated, chained, or compared
    across methods (stage_anchored_trajectory, the 7.5 RANSAC summary,
    plot_phase_amp_fit_grid's returned fit dicts). Do NOT apply it to the
    dz_est passed into plot_phase_slice/diag['fitted'] overlays -- those must
    stay in the image's own native row convention to correctly reproduce the
    plotted phase data."""
    return dz_est * DZ_SIGN_BY_METHOD[method]


# Per-technique WLS/RANSAC amplitude-weight exponent + automatic k-space gate,
# ported from FieldData_Playground.ipynb cell [64]'s WLS_FIT_DEFAULTS_BY_METHOD.
# Only wls_pow matters for k-space (napari-painted) picks -- kz_band_fac/
# kx_band_fac/amp_thr are ignored whenever an explicit mask is given. All four
# matter for b-scan (amplitude-domain) picks, where the (kz, kx) cell
# selection is still automatic (built by _auto_kx_mask, 7.4); painting there
# only sets the spatial crop and softened window. The old notebook's per-pair
# band overrides (MANUAL_KZ_BAND_FAC_BY_METHOD / MANUAL_KX_BAND_FAC_BY_METHOD)
# were tuned against the now-retired rectangular-window ROI and are not
# carried forward -- these per-method defaults are a starting point, not yet
# re-tuned against the new napari-only pipeline.
WLS_FIT_DEFAULTS_BY_METHOD = {
    "gazdag":       dict(wls_pow=3, kz_band_fac=0.5, kx_band_fac=2.0, amp_thr=0.20),
    "kirchhoff_bp": dict(wls_pow=3, kz_band_fac=0.5, kx_band_fac=2.0, amp_thr=0.20),
    "backprop":     dict(wls_pow=3, kz_band_fac=0.5, kx_band_fac=1.5, amp_thr=0.10),
}


def _kspace_display_crop(kz_ax, kx_ax, kz_cent, kdisp_fac=3.0, pos_kx_only=True):
    """Index range keeping |kz| < kdisp_fac*kz_cent (and, by default, only
    kx >= 0) on an fftshifted (kz_ax, kx_ax) grid -- the crop every k-space
    display panel in this notebook (picking below, 7.4's workflow figure,
    7.5's diagnostics) uses, so a painted/fitted mask and its displayed
    phase/amplitude always line up pixel-for-pixel.

    pos_kx_only=True (default) restricts the crop to kx >= 0 -- NOT kz (an
    earlier version of this function restricted kz instead; see
    restricting_kx.md at the repo root for the full reasoning -- that was
    backwards). For this borehole/VRP geometry the antenna radiates along x
    (radial/time), so x -- not z -- is the carrier/wavelet axis: base/mon
    being real-valued makes the cross-spectrum exactly Hermitian
    (XS(-kz,-kx) = conj(XS(kz,kx))), and with the carrier on x the redundant
    mirror copy sits at -kx, not -kz. z is the along-borehole spatial axis:
    a genuine target's diffraction response has a real, independent +kz
    flank (antenna receding) and -kz flank (antenna approaching) --
    restricting kz>0 instead deletes half of every real target's structure,
    not a mathematical duplicate. Fitting phi = kz*dz + kx*dx + phi_0
    through both the +kx and -kx mirror lobes at once forces a single shared
    phi_0 onto data that actually needs phi_0 and -phi_0, biasing the fitted
    plane -- restricting what's even shown/paintable here is what enforces
    "positive kx only" for a hand-painted k-space mask, simpler and more
    robust than trusting the painter to avoid the mirror by eye. Returns
    (iz0, iz1, ix0, ix1, kz_disp, kx_disp)."""
    kz_disp, kx_disp = np.fft.fftshift(kz_ax), np.fft.fftshift(kx_ax)
    if pos_kx_only:
        ix = np.where((kx_disp >= 0) & (kx_disp < kdisp_fac * kz_cent))[0]
    else:
        ix = np.where(np.abs(kx_disp) < kdisp_fac * kz_cent)[0]
    iz = np.where(np.abs(kz_disp) < kdisp_fac * kz_cent)[0]
    return int(iz.min()), int(iz.max()) + 1, int(ix.min()), int(ix.max()) + 1, kz_disp, kx_disp

In [ ]:
def launch_napari_picker(image, kind, name, *, extra_image=None, extra_name=None):
    """kind: 'bscan' (paint spatial ROI on the difference image) or
    'kspace' (paint directly on the cross-spectrum amplitude/envelope).
    Consolidates the two independently-duplicated inline napari.Viewer() call
    sites in FieldData_Playground.ipynb (cells [70]-[72] and [73]-[75]) into
    one reusable pair with harvest_napari_mask() below.

    extra_image/extra_name adds a second, initially-hidden toggle layer
    (e.g. cross-spectrum phase alongside the k-space amplitude/envelope
    image) -- purely an optional viewing aid, never itself painted on. A
    blended-visible version of this was tried and reverted: painting against
    two simultaneously-visible images made picking harder to judge, not
    easier, so the single primary image (envelope for k-space, the
    difference B-scan for b-scan) is what's shown by default -- toggle the
    extra layer on in napari's own layer list if you want it.

    Auto-selects and arms the mask layer for painting so there's nothing to
    click before picking can start."""
    import napari
    viewer = napari.Viewer(title=f"{name} ({kind})")
    viewer.add_image(image, colormap="inferno" if kind == "kspace" else "RdBu_r")
    if extra_image is not None:
        viewer.add_image(extra_image, name=extra_name, colormap="twilight", visible=False)
    mask_layer = viewer.add_labels(np.zeros(image.shape, dtype=int), name="manual_mask")
    viewer.layers.selection.active = mask_layer
    mask_layer.mode = "paint"
    mask_layer.brush_size = max(1, min(image.shape) // 20)
    return viewer, mask_layer


def harvest_napari_mask(mask_layer, gaussian_soften_px=None):
    """Read back the painted mask; Gaussian-soften edges for the bscan
    domain only, since a hard-edged spatial mask leaks spectral energy the
    same way an imprecise k-space pick does (chapter 7.3's sign-reversal
    failure mode)."""
    mask = mask_layer.data.astype(bool)
    if gaussian_soften_px:
        mask = gaussian_filter(mask.astype(float), gaussian_soften_px) > 0.5
    return mask

In [ ]:
# TODO (interactive): wavenumber-domain (k-space) picking -- one Launch/Harvest
# session per (method, pair), 3 methods x 4 stage pairs = 12 sessions total.
# Paint directly on the cross-spectrum amplitude; the fit is then restricted to
# exactly the painted cells (mask=..., no amplitude threshold or band needed).
#
# Run the Launch / Harvest cell pair below once per queue entry (x12). napari
# is interactive and can't run to completion inside a single non-interactive
# cell -- same reason FieldData_Playground.ipynb's picking sections (cells
# [100]-[112]) split into a Launch/Harvest pair too. KSPACE_PICK_IDX tracks
# progress and auto-advances; kspace_masks persists across reruns of this
# setup cell (the `if ... not in globals()` guards below), so you can pick a
# few, come back later, and pick the rest without losing progress. The
# Harvest cell also saves kspace_masks to KSPACE_PICKS_FILE after every pick,
# so progress survives a kernel restart too -- set LOAD_SAVED_KSPACE_PICKS =
# False below to redo every pick from scratch instead of keeping saved ones.
KSPACE_PICK_METHODS = ("kirchhoff_bp", "gazdag", "backprop")
KSPACE_PICK_QUEUE = [(method, run_a, run_b)
                      for run_a, run_b in STAGE_PAIRS
                      for method in KSPACE_PICK_METHODS]

KSPACE_PICKS_FILE = OUT_DIR / "kspace_picks.pkl"
LOAD_SAVED_KSPACE_PICKS = True  # False = ignore any saved file and re-pick everything

if "kspace_masks" not in globals():
    if LOAD_SAVED_KSPACE_PICKS and KSPACE_PICKS_FILE.exists():
        with open(KSPACE_PICKS_FILE, "rb") as f:
            kspace_masks = pickle.load(f)
        print(f"Loaded {len(kspace_masks)} saved k-space picks from {KSPACE_PICKS_FILE}.")
    else:
        kspace_masks = {}       # {(method, run_a, run_b): bool mask, padded non-fftshifted (kz, kx) grid}
if "KSPACE_PICK_IDX" not in globals():
    KSPACE_PICK_IDX = next((i for i, key in enumerate(KSPACE_PICK_QUEUE) if key not in kspace_masks),
                           len(KSPACE_PICK_QUEUE))  # resume after the last loaded/picked entry

print(f"{len(KSPACE_PICK_QUEUE)} (method, pair) k-space picks queued.")
_next = "done" if KSPACE_PICK_IDX >= len(KSPACE_PICK_QUEUE) else KSPACE_PICK_QUEUE[KSPACE_PICK_IDX]
print(f"{len(kspace_masks)} already picked; resuming at index {KSPACE_PICK_IDX} ({_next}).")

In [ ]:
# TODO (interactive) -- 1: launch napari for the next (method, pair) k-space
# pick. Paint the 'manual_mask' layer (paintbrush, label 1 -- already selected
# and armed) over the k-cells you want in the fit; leave everything else at 0.
# Viewer opens on the cross-spectrum amplitude/envelope (a blended phase+
# amplitude overlay was tried and reverted -- it made picking harder to judge,
# not easier); toggle the hidden 'phase (rad)' layer on in napari's layer
# list if you want to cross-check phase directly. Run the Harvest cell once
# done.
if KSPACE_PICK_IDX >= len(KSPACE_PICK_QUEUE):
    raise RuntimeError(f"All {len(KSPACE_PICK_QUEUE)} k-space picks done -- see kspace_masks "
                        "and 7.4/7.5 below. Set KSPACE_PICK_IDX = 0 to redo picks.")

kspace_method, kspace_run_a, kspace_run_b = KSPACE_PICK_QUEUE[KSPACE_PICK_IDX]
kspace_kz_cent = KZ_CENT_BY_METHOD[kspace_method]
kspace_dz, kspace_dx = DZ_BY_METHOD[kspace_method], DX_BY_METHOD[kspace_method]
kspace_base = post_imaged[kspace_method][kspace_run_a]
kspace_mon = post_imaged[kspace_method][kspace_run_b]

# No spatial ROI/window here -- napari picks the k-cells directly, so a Tukey
# taper over the full image only controls edge leakage into the FFT (the
# retired rectangular-window ROI is not reintroduced here).
_, _, _, _, kspace_diag = wls_phase_plane_fit(
    kspace_base, kspace_mon, dz=kspace_dz, dx=kspace_dx, kz_cent=kspace_kz_cent,
    taper="tukey", return_diagnostics=True)

KSPACE_IZ0, KSPACE_IZ1, KSPACE_IX0, KSPACE_IX1, _, _ = _kspace_display_crop(
    kspace_diag["kz_ax"], kspace_diag["kx_ax"], kspace_kz_cent)

w_shift = np.fft.fftshift(kspace_diag["w"])[KSPACE_IZ0:KSPACE_IZ1, KSPACE_IX0:KSPACE_IX1]
phi_shift = np.fft.fftshift(kspace_diag["phi"])[KSPACE_IZ0:KSPACE_IZ1, KSPACE_IX0:KSPACE_IX1]

kspace_viewer, kspace_mask_layer = launch_napari_picker(
    w_shift, "kspace", name=f"{kspace_method} {kspace_run_a}->{kspace_run_b}",
    extra_image=phi_shift, extra_name="phase (rad)")

print(f"[{KSPACE_PICK_IDX + 1}/{len(KSPACE_PICK_QUEUE)}] {kspace_method} "
      f"prof_{kspace_run_a}->{kspace_run_b}: k-space crop "
      f"{w_shift.shape[0]}x{w_shift.shape[1]} px "
      f"(|kz| < 3*kz_cent = {3 * kspace_kz_cent:.1f} rad/m, 0 <= kx < 3*kz_cent -- "
      "positive kx only, see restricting_kx.md). Paint 'manual_mask', then run the Harvest cell.")

In [ ]:
# TODO (interactive) -- 2: harvest the k-space pick, store it, advance the queue.
mask_crop = harvest_napari_mask(kspace_mask_layer)
n_painted = int(mask_crop.sum())
print(f"{n_painted} k-cells painted for {kspace_method} ({kspace_run_a}->{kspace_run_b}).")
if n_painted < 3:
    raise RuntimeError("Paint at least 3 k-cells in 'manual_mask' (kspace_viewer, Launch cell) "
                        "before running this cell.")

mask_shift_full = np.zeros(kspace_diag["w"].shape, dtype=bool)
mask_shift_full[KSPACE_IZ0:KSPACE_IZ1, KSPACE_IX0:KSPACE_IX1] = mask_crop
kspace_masks[(kspace_method, kspace_run_a, kspace_run_b)] = np.fft.ifftshift(mask_shift_full)

KSPACE_PICKS_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(KSPACE_PICKS_FILE, "wb") as f:
    pickle.dump(kspace_masks, f)

KSPACE_PICK_IDX += 1
if KSPACE_PICK_IDX < len(KSPACE_PICK_QUEUE):
    _nm, _na, _nb = KSPACE_PICK_QUEUE[KSPACE_PICK_IDX]
    print(f"-- {KSPACE_PICK_IDX}/{len(KSPACE_PICK_QUEUE)} done. Next: {_nm} ({_na}->{_nb}). "
          "Re-run the Launch cell above.")
else:
    print(f"-- All {len(KSPACE_PICK_QUEUE)} k-space picks done (saved to {KSPACE_PICKS_FILE}). "
          "Continue to the B-scan picking below.")

In [ ]:
# TODO (interactive): amplitude-domain (B-scan) picking -- the counterpart to
# the k-space picking above. Painting happens on the spatial difference image
# directly, tracing the reflector's actual shape; the paint only sets the
# spatial crop and a Gaussian-softened window, the (kz, kx) cell selection
# inside it is still the automatic band + amplitude gate
# (WLS_FIT_DEFAULTS_BY_METHOD) -- so this is a genuinely different test of the
# ROI's sensitivity than the hand-curated k-space picks above.
#
# Same Launch/Harvest/queue structure as the k-space section: run the pair
# below once per queue entry (x12); BSCAN_PICK_IDX auto-advances, bscan_masks
# persists across reruns of this setup cell, and the Harvest cell saves it to
# BSCAN_PICKS_FILE so progress survives a kernel restart too -- set
# LOAD_SAVED_BSCAN_PICKS = False below to redo every pick from scratch
# instead of keeping saved ones.
BSCAN_PICK_QUEUE = [(method, run_a, run_b)
                     for run_a, run_b in STAGE_PAIRS
                     for method in KSPACE_PICK_METHODS]

BSCAN_PICKS_FILE = OUT_DIR / "bscan_picks.pkl"
LOAD_SAVED_BSCAN_PICKS = True  # False = ignore any saved file and re-pick everything

if "bscan_masks" not in globals():
    if LOAD_SAVED_BSCAN_PICKS and BSCAN_PICKS_FILE.exists():
        with open(BSCAN_PICKS_FILE, "rb") as f:
            bscan_masks = pickle.load(f)
        print(f"Loaded {len(bscan_masks)} saved B-scan picks from {BSCAN_PICKS_FILE}.")
    else:
        bscan_masks = {}       # {(method, run_a, run_b): dict(full_mask, z0, z1, x0, x1, soft_win)}
if "BSCAN_PICK_IDX" not in globals():
    BSCAN_PICK_IDX = next((i for i, key in enumerate(BSCAN_PICK_QUEUE) if key not in bscan_masks),
                          len(BSCAN_PICK_QUEUE))  # resume after the last loaded/picked entry

print(f"{len(BSCAN_PICK_QUEUE)} (method, pair) B-scan picks queued.")
_next = "done" if BSCAN_PICK_IDX >= len(BSCAN_PICK_QUEUE) else BSCAN_PICK_QUEUE[BSCAN_PICK_IDX]
print(f"{len(bscan_masks)} already picked; resuming at index {BSCAN_PICK_IDX} ({_next}).")

In [ ]:
# TODO (interactive) -- 1: launch napari for the next (method, pair) B-scan
# pick. Paint the 'manual_mask' layer over the real reflection change in the
# difference image; run the Harvest cell once done. Viewer opens on the plain
# difference B-scan (a blended energy-envelope overlay was tried and
# reverted -- it made picking harder to judge, not easier).
if BSCAN_PICK_IDX >= len(BSCAN_PICK_QUEUE):
    raise RuntimeError(f"All {len(BSCAN_PICK_QUEUE)} B-scan picks done -- see bscan_masks "
                        "and 7.4/7.5 below. Set BSCAN_PICK_IDX = 0 to redo picks.")

bscan_method, bscan_run_a, bscan_run_b = BSCAN_PICK_QUEUE[BSCAN_PICK_IDX]
bscan_diff = differences[bscan_method][(bscan_run_a, bscan_run_b)]

bscan_viewer, bscan_mask_layer = launch_napari_picker(
    bscan_diff, "bscan", name=f"{bscan_method} {bscan_run_a}->{bscan_run_b}")

print(f"[{BSCAN_PICK_IDX + 1}/{len(BSCAN_PICK_QUEUE)}] {bscan_method} "
      f"prof_{bscan_run_a}->{bscan_run_b}: difference image "
      f"{bscan_diff.shape[0]}x{bscan_diff.shape[1]} px. Paint 'manual_mask' over "
      "the real reflection change, then run the Harvest cell.")

In [ ]:
# TODO (interactive) -- 2: harvest the pick, crop + soften the painted mask
# into a spatial window, store, advance. Mirrors FieldData_Playground.ipynb's
# B-Scan RANSAC Trial harvest cell: crop to the painted bounding box (+ a
# margin so the Gaussian softening has real data to blend into), soften the
# hard painted edge, then store both the raw full-image mask (for the 7.4
# workflow figure) and the windowed crop (for the fit itself, via
# _fit_inputs).
BSCAN_GAUSS_SIGMA_PX = 3.0

full_mask = harvest_napari_mask(bscan_mask_layer)
n_painted = int(full_mask.sum())
print(f"{n_painted} image px painted for {bscan_method} ({bscan_run_a}->{bscan_run_b}).")
if n_painted < 20:
    raise RuntimeError("Paint a larger region in 'manual_mask' (bscan_viewer, Launch cell) "
                        "before running this cell.")

rows, cols = np.where(full_mask)
z0, z1 = int(rows.min()), int(rows.max()) + 1
x0, x1 = int(cols.min()), int(cols.max()) + 1
margin = int(np.ceil(3 * BSCAN_GAUSS_SIGMA_PX))
z0p, z1p = max(0, z0 - margin), min(bscan_diff.shape[0], z1 + margin)
x0p, x1p = max(0, x0 - margin), min(bscan_diff.shape[1], x1 + margin)

soft_win = gaussian_filter(full_mask[z0p:z1p, x0p:x1p].astype(float), BSCAN_GAUSS_SIGMA_PX)
if soft_win.max() > 0:
    soft_win = soft_win / soft_win.max()

bscan_masks[(bscan_method, bscan_run_a, bscan_run_b)] = dict(
    full_mask=full_mask, z0=z0p, z1=z1p, x0=x0p, x1=x1p, soft_win=soft_win)

BSCAN_PICKS_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(BSCAN_PICKS_FILE, "wb") as f:
    pickle.dump(bscan_masks, f)

BSCAN_PICK_IDX += 1
if BSCAN_PICK_IDX < len(BSCAN_PICK_QUEUE):
    _nm, _na, _nb = BSCAN_PICK_QUEUE[BSCAN_PICK_IDX]
    print(f"-- {BSCAN_PICK_IDX}/{len(BSCAN_PICK_QUEUE)} done. Next: {_nm} ({_na}->{_nb}). "
          "Re-run the Launch cell above.")
else:
    print(f"-- All {len(BSCAN_PICK_QUEUE)} B-scan picks done (saved to {BSCAN_PICKS_FILE}). "
          "Continue to 7.4 below.")

## 7.4 WLS Cross-Spectrum Phase Plane Fit

Stage-anchored chaining is the *sole* trajectory strategy: one representative
pair per stage, fit independently, chained by cumulative sum since the pairs are
already end-to-end consecutive.

**Discarded from `FieldData_Playground.ipynb`:** `helper_functions/WLS.py`'s
`estimate_shift_2d` (not imported here at all -- other notebooks still depend on
that file, so it is left alone, just unused by this pipeline); the notebook-inline
`_estimate_shift_2d` (cell [64], a third parallel implementation); Strategy 1
/ consecutive-increment chaining (cell [82]) and Strategy 2 / fixed-baseline
chaining (cell [84]); Strategy 3's separate local-anchor-plus-boundary-pair
construction (cell [86]) -- what the chapter text calls "stage-anchored" is the
notebook's simpler "Strategy 3b" (cell [88]), which is the only version kept here.

In [ ]:
def _auto_kx_mask(base, mon, dz, dx, kz_cent, kz_band_fac, kx_band_fac, amp_thr, taper):
    """Automatic (kz, kx)-cell gate, restricted to kx > 0 -- NOT
    wls_phase_plane_fit/ransac_phase_plane_fit's own kz > 0 default. See
    restricting_kx.md at the repo root: for this borehole/VRP geometry the
    antenna radiates along x (radial/time), so x -- not z -- is the
    carrier/wavelet axis, and it's the +-kx lobes that are the exact
    Hermitian mirror ghost (XS(-kz,-kx) = conj(XS(kz,kx)), redundant copy at
    -kx). z is the along-borehole spatial axis: a real target's diffraction
    response has a genuine, independent +kz flank (antenna receding) and
    -kz flank (antenna approaching) -- restricting kz>0 instead (what the
    library's own pos_kz_only default does) deletes real structure, not a
    mathematical duplicate.

    Computed by first getting the full (unmasked) cross-spectrum
    diagnostics from a throwaway wls_phase_plane_fit call -- same taper as
    the real fit that follows, so the cross-spectrum matches exactly --
    then building the band + amplitude gate ourselves with the restriction
    on the other axis, and handing it back as an explicit mask=, which
    bypasses wls_phase_plane_fit/ransac_phase_plane_fit's own pos_kz_only
    logic entirely (mask=... always overrides it)."""
    _, _, _, _, diag = wls_phase_plane_fit(
        base, mon, dz, dx, kz_cent, taper=taper, return_diagnostics=True)
    KZ, KX, w = diag["KZ"], diag["KX"], diag["w"]
    band = (np.abs(KZ) < kz_band_fac * kz_cent) & (np.abs(KX) < kx_band_fac * kz_cent) & (KX > 0)
    return (w > amp_thr * w.max()) & band


def _fit_inputs(method, run_a, run_b, domain):
    """Resolve (base, mon, dz, dx, kz_cent, fit_kwargs, wls_pow) for one
    (method, pair, domain) combination, from whichever picking store 7.3 left
    behind. The two picking domains store fundamentally different things --
    kspace_masks holds a boolean mask directly on the padded (kz, kx) grid
    (the pick IS the fit population, mask=... in wls_phase_plane_fit /
    ransac_phase_plane_fit); bscan_masks holds a spatial crop + softened
    window (the pick only fixes the spatial ROI, the (kz, kx) selection
    still needs an automatic gate) -- this is the one place that difference
    is resolved, shared by stage_anchored_trajectory, build_wls_workflow_
    figure, plot_phase_amp_fit_grid, and the RANSAC summary loop below.

    Both domains now return an explicit mask= (never kz_band_fac/
    kx_band_fac/amp_thr passed straight through for the library's own
    automatic gate to build): its pos_kz_only default restricts the wrong
    axis for this borehole geometry (restricting_kx.md), so the b-scan-
    domain gate is built here instead, by _auto_kx_mask, restricted to
    kx > 0.

    wls_pow is returned separately (not folded into fit_kwargs) since
    wls_phase_plane_fit calls it `wls_pow` but ransac_phase_plane_fit calls
    the same concept `final_wls_pow` -- callers pass whichever name their
    fit function expects."""
    base_full = post_imaged[method][run_a]
    mon_full = post_imaged[method][run_b]
    dz, dx, kz_cent = DZ_BY_METHOD[method], DX_BY_METHOD[method], KZ_CENT_BY_METHOD[method]
    defaults = WLS_FIT_DEFAULTS_BY_METHOD[method]

    if domain == "kspace":
        base, mon = base_full, mon_full
        kwargs = dict(taper="tukey", mask=kspace_masks[(method, run_a, run_b)])
    elif domain == "bscan":
        crop = bscan_masks[(method, run_a, run_b)]
        z0, z1, x0, x1 = crop["z0"], crop["z1"], crop["x0"], crop["x1"]
        base = base_full[z0:z1, x0:x1] * crop["soft_win"]
        mon = mon_full[z0:z1, x0:x1] * crop["soft_win"]
        kx_mask = _auto_kx_mask(base, mon, dz, dx, kz_cent, defaults["kz_band_fac"],
                                 defaults["kx_band_fac"], defaults["amp_thr"], taper="none")
        kwargs = dict(taper="none", mask=kx_mask)
    else:
        raise ValueError(f"domain must be 'kspace' or 'bscan', got {domain!r}")

    return base, mon, dz, dx, kz_cent, kwargs, defaults["wls_pow"]

In [ ]:
def stage_anchored_trajectory(method, stage_pairs=STAGE_PAIRS, mask_domain="kspace"):
    """One WLS fit per stage-representative pair; chain by cumulative sum.
    No boundary/bridging pairs needed -- the pairs are already end-to-end
    consecutive (profile n of pair i is profile 1 of pair i+1's stage).
    dz_est is sign-corrected via _physical_dz before accumulating/storing
    (back-propagation's row convention is opposite Kirchhoff/Gazdag's -- see
    DZ_SIGN_BY_METHOD above), so every caller of this function always gets a
    physically-comparable dz/dz_step regardless of method."""
    cum_dx = cum_dz = 0.0
    trajectory = []
    for run_a, run_b in stage_pairs:
        base, mon, dz, dx, kz_cent, kwargs, wls_pow = _fit_inputs(method, run_a, run_b, mask_domain)
        dz_est, dx_est, phi_0, n_mask = wls_phase_plane_fit(
            base, mon, dz, dx, kz_cent, wls_pow=wls_pow, **kwargs)
        dz_est_phys = _physical_dz(method, dz_est)
        cum_dx += dx_est
        cum_dz += dz_est_phys
        trajectory.append(dict(pair=(run_a, run_b), dx=cum_dx, dz=cum_dz,
                                dx_step=dx_est, dz_step=dz_est_phys, phi_0=phi_0, n_mask=n_mask))
    return trajectory


gazdag_trajectory = stage_anchored_trajectory("gazdag")

In [ ]:
def build_wls_workflow_figure(method, run_a, run_b, kdisp_fac=2.0):
    """Assemble Figure 7.4: 2x2 -- row 1 the two migrated images being
    compared, row 2 the amplitude-domain and wavenumber-domain napari picks.
    Does not exist in the old notebook; built here from pieces of the napari
    k-space cell [71]-[72] and B-scan cell [74]-[75].

    Recomputes the cross-spectrum phase on the SAME padded (kz, kx) grid the
    k-space mask was painted on (taper='tukey', no roi_px -- matches 7.3's
    k-space Launch cell), so the mask contour lines up with the phase panel
    pixel-for-pixel instead of against an unrelated unpadded fft2(diff).

    kdisp_fac=2.0 (vs 3.0 used for picking/diagnostics) crops the k-space
    panel tighter -- still comfortably >= every WLS_FIT_DEFAULTS_BY_METHOD
    kx_band_fac (max 2.0), just without the extra empty/low-amplitude margin
    that made the panel look cluttered. The phase panel is also rendered
    with per-pixel alpha tied to amplitude (WLS_FIT_DEFAULTS_BY_METHOD
    [method]['amp_thr']), so phase outside the high-amplitude region fades
    instead of visually competing with the region that's actually meaningful
    -- independent of the picked mask itself, so you can see whether the
    green pick contour actually tracks the high-amplitude (opaque) area."""
    base, mon = post_imaged[method][run_a], post_imaged[method][run_b]
    diff = differences[method][(run_a, run_b)]
    dz, dx, kz_cent = DZ_BY_METHOD[method], DX_BY_METHOD[method], KZ_CENT_BY_METHOD[method]
    kspace_mask = kspace_masks[(method, run_a, run_b)]
    bscan_mask = bscan_masks[(method, run_a, run_b)]["full_mask"]

    # Depth/radial-distance axes for the profile & B-scan panels: kirchhoff_bp/
    # gazdag share the fixed DEPTH_AXIS/X_IMG grid for every run; back-propagation
    # has its own true-metre grid per run (backprop_axes), with the opposite
    # row-index convention documented above KZ_CENT_BY_METHOD. The same
    # [x[0], x[-1], y[-1], y[0]] extent formula labels either convention
    # correctly since it only assumes row i of the image corresponds to y[i].
    if method == "backprop":
        depth_a, radial_a = backprop_axes[run_a]
        depth_b, radial_b = backprop_axes[run_b]
    else:
        depth_a = depth_b = DEPTH_AXIS
        radial_a = radial_b = X_IMG
    extent_a = [radial_a[0], radial_a[-1], depth_a[-1], depth_a[0]]
    extent_b = [radial_b[0], radial_b[-1], depth_b[-1], depth_b[0]]
    extent_diff = extent_a  # diff = mon - base is base-shaped; label it with run_a's axes

    _, _, _, _, diag = wls_phase_plane_fit(base, mon, dz, dx, kz_cent, taper="tukey", return_diagnostics=True)
    iz0, iz1, ix0, ix1, kz_disp, kx_disp = _kspace_display_crop(
        diag["kz_ax"], diag["kx_ax"], kz_cent, kdisp_fac=kdisp_fac)
    phi_c = np.fft.fftshift(diag["phi"])[iz0:iz1, ix0:ix1]
    w_c = np.fft.fftshift(diag["w"])[iz0:iz1, ix0:ix1]
    mask_c = np.fft.fftshift(kspace_mask)[iz0:iz1, ix0:ix1]

    amp_thr = WLS_FIT_DEFAULTS_BY_METHOD[method]["amp_thr"]
    phi_rgba = plt.get_cmap("twilight")(plt.Normalize(-np.pi, np.pi)(phi_c))
    phi_rgba[..., 3] = np.where(w_c > amp_thr * w_c.max(), 1.0, 0.15)
    extent = [kx_disp[ix0], kx_disp[ix1 - 1], kz_disp[iz1 - 1], kz_disp[iz0]]

    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    # Shared symmetric colour scale across both profile panels so one colorbar
    # can represent the whole row (each panel previously auto-scaled independently).
    vmax_row1 = max(np.abs(base).max(), np.abs(mon).max())
    im00 = axes[0, 0].imshow(base, cmap="RdBu_r", extent=extent_a, aspect="auto",
                              vmin=-vmax_row1, vmax=vmax_row1)
    axes[0, 0].set_title(f"Profile {run_a} ({method})")
    axes[0, 0].set_xlabel("Radial distance from borehole [m]")
    axes[0, 0].set_ylabel("Depth [m]")
    axes[0, 1].imshow(mon, cmap="RdBu_r", extent=extent_b, aspect="auto",
                       vmin=-vmax_row1, vmax=vmax_row1)
    axes[0, 1].set_title(f"Profile {run_b} ({method})")
    axes[0, 1].set_xlabel("Radial distance from borehole [m]")
    axes[0, 1].set_ylabel("Depth [m]")
    # cb_amp = fig.colorbar(im00, ax=[axes[0, 0], axes[0, 1]], fraction=0.046, pad=0.02)
    # cb_amp.set_label("Amplitude [a.u.]")
    axes[1, 0].imshow(diff, cmap="RdBu_r", extent=extent_diff, aspect="auto")
    axes[1, 0].contour(bscan_mask.astype(float), levels=[0.5], colors="lime", linewidths=1,
                        extent=[extent_diff[0], extent_diff[1], extent_diff[3], extent_diff[2]])
    axes[1, 0].set_title("Amplitude-domain (B-scan) manual pick")
    axes[1, 0].set_xlabel("Radial distance from borehole [m]")
    axes[1, 0].set_ylabel("Depth [m]")
    axes[1, 1].imshow(phi_rgba, extent=extent, aspect="auto")
    axes[1, 1].contour(kx_disp[ix0:ix1], kz_disp[iz0:iz1], mask_c.astype(float),
                        levels=[0.5], colors="lime", linewidths=1)
    axes[1, 1].set_title("Wavenumber-domain (k-space) manual pick")
    axes[1, 1].set_xlabel("$k_x$ [rad/m]")
    axes[1, 1].set_ylabel("$k_z$ [rad/m]")
    axes[1, 1].set_ylim(4, -4)  # zoom in on |k_z| <= 4 rad/m

    # phi_rgba is pre-composited RGBA (per-pixel alpha tied to amplitude), so the
    # image itself carries no reusable scalar mapping -- a standalone
    # ScalarMappable is what fig.colorbar needs instead.
    phase_sm = plt.cm.ScalarMappable(cmap=plt.get_cmap("twilight"), norm=plt.Normalize(-np.pi, np.pi))
    phase_sm.set_array([])
    cb_phase = fig.colorbar(phase_sm, ax=axes[1, 1], fraction=0.046)
    cb_phase.set_label("Phase [rad]")

    fig.tight_layout()
    return fig


fig = build_wls_workflow_figure("gazdag", 1, 3)
save_fig(fig, "wls_workflow_1_to_3", study=STUDY, prefix="FD_", category="Compilations")

### Back-Propagation Cross-Check and Stage Displacements

Same napari-picked workflow applied to the corrected back-propagation focus
frames, with the depth-axis sign correction (back-propagation's row index runs
opposite to Kirchhoff/Gazdag) folded in before the fit.

In [ ]:
import pandas as pd

# NOTE: no separate BP_DEPTH_SIGN correction needed here -- stage_anchored_trajectory
# already applies _physical_dz internally (see DZ_SIGN_BY_METHOD above), so
# backprop_trajectory's dz/dz_step are already sign-corrected. Multiplying by
# a sign again here would silently flip them back to the wrong (raw) sign.
backprop_trajectory = stage_anchored_trajectory("backprop")

stage_table = pd.DataFrame({
    "stage": STAGE_NAMES,
    "pair": STAGE_PAIRS,
    "dz_gazdag": [s["dz"] for s in gazdag_trajectory],
    "dx_gazdag": [s["dx"] for s in gazdag_trajectory],
    "dz_backprop": [s["dz"] for s in backprop_trajectory],
    "dx_backprop": [s["dx"] for s in backprop_trajectory],
})
stage_table.to_csv(OUT_DIR / "roi_phase" / "fielddata_stage_displacements.csv", index=False)
stage_table

## 7.5 Interpretation (WLS vs. RANSAC)

RANSAC robustness check, run for all four representative pairs, all three
migration techniques, and both napari picking domains (wavenumber, amplitude).

In [ ]:
# from matplotlib.patches import Rectangle
# from matplotlib.lines import Line2D


# def plot_phase_amp_fit_grid(method, run_a, run_b, domain, stage_name, kdisp_fac=2.0):
#     """Assemble Figure 7.5-1: 2x4 -- columns cross-spectrum phase /
#     cross-spectrum amplitude / x-direction (kx) fit / z-direction (kz) fit;
#     rows WLS / RANSAC, both refit on the exact same picked pixels (mask/gate
#     resolved once by _fit_inputs) so the fitting method is the only
#     difference between the two estimates. Extends the old notebook's
#     plot_phase_slice / cell [104] diagnostic (2x3: phase, kz-fit, kx-fit --
#     no amplitude column) with a genuinely new amplitude panel.

#     kdisp_fac=2.0 (vs 3.0 used elsewhere) crops the phase/amplitude panels
#     tighter -- still comfortably >= every WLS_FIT_DEFAULTS_BY_METHOD
#     kx_band_fac (max 2.0), so the automatic (b-scan-domain) gate's full
#     population always stays inside frame, just without the extra empty/
#     low-amplitude margin that made the panel look cluttered. The phase
#     panels are rendered with per-pixel alpha tied to amplitude
#     (WLS_FIT_DEFAULTS_BY_METHOD[method]['amp_thr']), fading phase outside the
#     high-amplitude region instead of it visually competing with the region
#     that's actually meaningful -- independent of the WLS/RANSAC population
#     contour, so you can see whether that contour actually tracks the
#     high-amplitude (opaque) area.

#     Both the phase and amplitude panels also get a dashed white rectangle
#     showing WLS_FIT_DEFAULTS_BY_METHOD[method]'s kz_band_fac/kx_band_fac band
#     (|kz| < kz_band_fac*kz_cent, 0 <= kx < kx_band_fac*kz_cent -- the same
#     |KZ| < kz_band_fac*kz_cent & |KX| < kx_band_fac*kz_cent formula
#     wls_phase_plane_fit/ransac_phase_plane_fit use internally, restricted to
#     kx > 0 to match this notebook's kx-only convention, see
#     restricting_kx.md). For domain='bscan' this is exactly the band
#     _auto_kx_mask gated on (before the amp_thr threshold further trims it,
#     which is what the lime population contour actually shows); for
#     domain='kspace' the band was NOT applied -- the population is a manual
#     napari pick with no band restriction at all -- so the rectangle is drawn
#     only as a size reference against the automatic-gate band, not a claim
#     about what bounded that row's fit.

#     _fit_inputs already supplies an explicit mask= for both domains (see its
#     docstring / restricting_kx.md), so no pos_kz_only/pos_kx_only override is
#     needed at the wls_phase_plane_fit/ransac_phase_plane_fit call sites below
#     -- an explicit mask always bypasses that logic entirely regardless of its
#     value. _kspace_display_crop's own pos_kx_only default (kx >= 0) is what
#     keeps this figure's displayed range consistent with what was actually
#     fit.

#     The plotted fit lines (plot_phase_slice below) intentionally use the RAW,
#     un-sign-corrected dz_wls/dz_r -- they have to stay in the image's own
#     native row convention to actually overlay the plotted phase data. Only
#     the returned summary dicts apply _physical_dz, since those are the
#     numbers meant to be read/compared/reported as physical displacement."""
#     base, mon, dz, dx, kz_cent, kwargs, wls_pow = _fit_inputs(method, run_a, run_b, domain)

#     dz_wls, dx_wls, phi0_wls, n_wls, diag_wls = wls_phase_plane_fit(
#         base, mon, dz, dx, kz_cent, wls_pow=wls_pow, return_diagnostics=True, **kwargs)
#     dz_r, dx_r, phi0_r, n_r, diag_r = ransac_phase_plane_fit(
#         base, mon, dz, dx, kz_cent, final_wls_pow=wls_pow, n_iter=2000, residual_thr=0.35,
#         random_state=0, return_diagnostics=True, **kwargs)

#     iz0, iz1, ix0, ix1, kz_disp, kx_disp = _kspace_display_crop(
#         diag_wls["kz_ax"], diag_wls["kx_ax"], kz_cent, kdisp_fac=kdisp_fac)
#     extent = [kx_disp[ix0], kx_disp[ix1 - 1], kz_disp[iz1 - 1], kz_disp[iz0]]

#     # 1-D slice points come from the fit's own input population (identical
#     # for WLS/RANSAC here, since both were called with the same mask/gate
#     # kwargs) -- exactly what plot_phase_slice's docstring expects, not a
#     # broader row/column-averaged band.
#     pop = diag_wls["mask"]
#     KZp, KXp = diag_wls["KZ"][pop], diag_wls["KX"][pop]
#     PHIp, Wp = diag_wls["phi"][pop], diag_wls["w"][pop]
#     outlier_r = ~diag_r["inlier_mask"][pop]

#     phi_c = np.fft.fftshift(diag_wls["phi"])[iz0:iz1, ix0:ix1]
#     w_c = np.fft.fftshift(diag_wls["w"])[iz0:iz1, ix0:ix1]
#     pop_c = np.fft.fftshift(pop)[iz0:iz1, ix0:ix1]
#     inlier_c = np.fft.fftshift(diag_r["inlier_mask"])[iz0:iz1, ix0:ix1]

#     amp_thr = WLS_FIT_DEFAULTS_BY_METHOD[method]["amp_thr"]
#     phi_rgba = plt.get_cmap("twilight")(plt.Normalize(-np.pi, np.pi)(phi_c))
#     phi_rgba[..., 3] = np.where(w_c > amp_thr * w_c.max(), 1.0, 0.15)

#     # Reference band: same |KZ| < kz_band_fac*kz_cent & |KX| < kx_band_fac*kz_cent
#     # formula wls_phase_plane_fit/ransac_phase_plane_fit use internally to
#     # build their own automatic gate (kz_band_fac/kx_band_fac from
#     # WLS_FIT_DEFAULTS_BY_METHOD), restricted to kx > 0 here to match
#     # _auto_kx_mask/_kspace_display_crop's convention rather than the
#     # library's own symmetric-kx default (moot anyway once mask= is given).
#     band_defaults = WLS_FIT_DEFAULTS_BY_METHOD[method]
#     band_kz = band_defaults["kz_band_fac"] * kz_cent
#     band_kx = band_defaults["kx_band_fac"] * kz_cent

#     fig, axes = plt.subplots(2, 4, figsize=(16, 7))
#     for row, (label, contour_mask) in enumerate((("WLS", pop_c), ("RANSAC", inlier_c))):
#         axes[row, 0].imshow(phi_rgba, extent=extent, aspect="auto")
#         axes[row, 0].contour(kx_disp[ix0:ix1], kz_disp[iz0:iz1], contour_mask.astype(float),
#                               levels=[0.5], colors="lime", linewidths=1)
#         axes[row, 0].set_title("Cross-spectrum phase")
#         axes[row, 0].set_ylabel(label)

#         axes[row, 1].imshow(w_c, cmap="viridis", extent=extent, aspect="auto")
#         axes[row, 1].set_title("Cross-spectrum amplitude")

#         for ax in (axes[row, 0], axes[row, 1]):
#             ax.add_patch(Rectangle((0.0, -band_kz), band_kx, 2 * band_kz, fill=False,
#                                     edgecolor="white", linestyle="--", linewidth=1.3))

#     band_note = ("auto-gate (kz,kx) band" if domain == "bscan" else
#                  "auto-gate band (reference only -- this row's pick is manual)")
#     axes[0, 0].legend(handles=[
#         Line2D([], [], color="white", linestyle="--", linewidth=1.3, label=band_note),
#         Line2D([], [], color="lime", linewidth=1, label="fit population"),
#     ], loc="upper right", fontsize=7, framealpha=0.6)

#     color_wls, color_r = "tab:blue", "tab:orange"
#     plot_phase_slice(axes[0, 2], KXp, KZp, PHIp, Wp, dx_wls, dz_wls, phi0_wls,
#                       xlabel="kx [rad/m]", color=color_wls, line_label="WLS")
#     plot_phase_slice(axes[0, 3], KZp, KXp, PHIp, Wp, dz_wls, dx_wls, phi0_wls,
#                       xlabel="kz [rad/m]", color=color_wls, line_label="WLS")
#     plot_phase_slice(axes[1, 2], KXp, KZp, PHIp, Wp, dx_r, dz_r, phi0_r,
#                       xlabel="kx [rad/m]", color=color_r, outlier=outlier_r, line_label="RANSAC")
#     plot_phase_slice(axes[1, 3], KZp, KXp, PHIp, Wp, dz_r, dx_r, phi0_r,
#                       xlabel="kz [rad/m]", color=color_r, outlier=outlier_r, line_label="RANSAC")
#     axes[0, 2].set_title("x-direction fit"); axes[0, 3].set_title("z-direction fit")
#     axes[1, 2].set_title("x-direction fit"); axes[1, 3].set_title("z-direction fit")

#     fig.suptitle(f"{method}, {stage_name} ({run_a}→{run_b}), {domain} picking")
#     fig.tight_layout()
#     return fig, dict(dz=_physical_dz(method, dz_wls), dx=dx_wls, n=n_wls), \
#                 dict(dz=_physical_dz(method, dz_r), dx=dx_r, n=n_r)


# fig, wls_fit, ransac_fit = plot_phase_amp_fit_grid("kirchhoff_bp", 20, 38, "bscan", "Chasing")
# save_fig(fig, "ransac_vs_wls_phase_amp_fit_kirchhoff_bp_chasing_20_to_38",
#          study=STUDY, prefix="FD_", category="Compilations")


In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D


def plot_phase_amp_fit_grid_tuned(method, run_a, run_b, domain, stage_name, kdisp_fac=2.0, *,
                                   amp_thr=None, kz_band_fac=None, kx_band_fac=None,
                                   residual_thr=0.35, n_iter=2000):
    """Sandbox copy of plot_phase_amp_fit_grid (7.5, canonical cell) for trying
    per-call amp_thr/kz_band_fac/kx_band_fac/residual_thr without touching
    WLS_FIT_DEFAULTS_BY_METHOD.

    amp_thr/kz_band_fac/kx_band_fac (default None -> use
    WLS_FIT_DEFAULTS_BY_METHOD[method]) only affect domain='bscan': they're
    threaded into a fresh _auto_kx_mask(...) call, which is what actually
    builds this domain's (kz, kx) gate restricted to kx > 0 (see
    restricting_kx.md). They do NOT get passed straight through to
    wls_phase_plane_fit/ransac_phase_plane_fit as raw kwargs -- doing that
    (what the first draft of this cell did) drops the explicit mask=, so
    those functions fall back to their own automatic gate instead, which
    only restricts KZ > 0 (pos_kz_only=True default) and leaves KX
    symmetric -- exactly the wrong-axis restriction restricting_kx.md
    describes, letting negative-kx cells straight back into the fit. For
    domain='kspace' these three are no-ops: that domain's mask is a manual
    napari pick with no band restriction at all, so tuning the amplitude
    gate there means re-painting, not changing a number here.

    residual_thr/n_iter are plain RANSAC parameters, uncoupled from the
    masking bug above -- free to tune either way, no restriction logic
    involved.
    """
    base, mon, dz, dx, kz_cent, kwargs, wls_pow = _fit_inputs(method, run_a, run_b, domain)
    wls_pow = 6
    if domain == "bscan":
        defaults = WLS_FIT_DEFAULTS_BY_METHOD[method]
        kx_mask = _auto_kx_mask(
            base, mon, dz, dx, kz_cent,
            kz_band_fac if kz_band_fac is not None else defaults["kz_band_fac"],
            kx_band_fac if kx_band_fac is not None else defaults["kx_band_fac"],
            amp_thr if amp_thr is not None else defaults["amp_thr"],
            taper="none")
        kwargs = dict(taper="none", mask=kx_mask)

    dz_wls, dx_wls, phi0_wls, n_wls, diag_wls = wls_phase_plane_fit(
        base, mon, dz, dx, kz_cent, wls_pow=wls_pow, return_diagnostics=True, **kwargs)
    dz_r, dx_r, phi0_r, n_r, diag_r = ransac_phase_plane_fit(
        base, mon, dz, dx, kz_cent, final_wls_pow=wls_pow, n_iter=n_iter, residual_thr=residual_thr,
        random_state=0, return_diagnostics=True, **kwargs)

    iz0, iz1, ix0, ix1, kz_disp, kx_disp = _kspace_display_crop(
        diag_wls["kz_ax"], diag_wls["kx_ax"], kz_cent, kdisp_fac=kdisp_fac)
    extent = [kx_disp[ix0], kx_disp[ix1 - 1], kz_disp[iz1 - 1], kz_disp[iz0]]

    pop = diag_wls["mask"]
    KZp, KXp = diag_wls["KZ"][pop], diag_wls["KX"][pop]
    PHIp, Wp = diag_wls["phi"][pop], diag_wls["w"][pop]
    outlier_r = ~diag_r["inlier_mask"][pop]

    phi_c = np.fft.fftshift(diag_wls["phi"])[iz0:iz1, ix0:ix1]
    w_c = np.fft.fftshift(diag_wls["w"])[iz0:iz1, ix0:ix1]
    pop_c = np.fft.fftshift(pop)[iz0:iz1, ix0:ix1]
    inlier_c = np.fft.fftshift(diag_r["inlier_mask"])[iz0:iz1, ix0:ix1]

    disp_amp_thr = amp_thr if amp_thr is not None else WLS_FIT_DEFAULTS_BY_METHOD[method]["amp_thr"]
    phi_rgba = plt.get_cmap("twilight")(plt.Normalize(-np.pi, np.pi)(phi_c))
    phi_rgba[..., 3] = np.where(w_c > disp_amp_thr * w_c.max(), 1.0, 0.15)

    band_defaults = WLS_FIT_DEFAULTS_BY_METHOD[method]
    band_kz = (kz_band_fac if kz_band_fac is not None else band_defaults["kz_band_fac"]) * kz_cent
    band_kx = (kx_band_fac if kx_band_fac is not None else band_defaults["kx_band_fac"]) * kz_cent

    FS_SUPTITLE, FS_TITLE, FS_LABEL, FS_LEGEND, FS_TICK = 20, 16, 15, 12, 12

    fig, axes = plt.subplots(2, 4, figsize=(24, 11))
    phase_cmap = plt.get_cmap("twilight")
    phase_norm = plt.Normalize(-np.pi, np.pi)
    for row, (label, contour_mask) in enumerate((("WLS", pop_c), ("RANSAC", inlier_c))):
        axes[row, 0].imshow(phi_rgba, extent=extent, aspect="auto")
        axes[row, 0].contour(kx_disp[ix0:ix1], kz_disp[iz0:iz1], contour_mask.astype(float),
                              levels=[0.5], colors="lime", linewidths=1)
        axes[row, 0].set_title("Cross-spectrum phase", fontsize=FS_TITLE)
        axes[row, 0].set_xlabel("kx [rad/m]", fontsize=FS_LABEL)
        axes[row, 0].set_ylabel(f"{label}\nkz [rad/m]", fontsize=FS_LABEL)
        axes[row, 0].set_ylim(-extent[1]/2, extent[1]/2)
        phase_sm = plt.cm.ScalarMappable(cmap=phase_cmap, norm=phase_norm)
        phase_sm.set_array([])
        cb_phase = fig.colorbar(phase_sm, ax=axes[row, 0], fraction=0.046)
        cb_phase.set_label("Phase [rad]", fontsize=FS_LABEL)
        cb_phase.ax.tick_params(labelsize=FS_TICK)

        im_amp = axes[row, 1].imshow(w_c, cmap="viridis", extent=extent, aspect="auto")
        axes[row, 1].set_title("Cross-spectrum amplitude", fontsize=FS_TITLE)
        axes[row, 1].set_xlabel("kx [rad/m]", fontsize=FS_LABEL)
        axes[row, 1].set_ylabel("kz [rad/m]", fontsize=FS_LABEL)
        axes[row, 1].set_ylim(-extent[1]/2, extent[1]/2)
        cb_amp = fig.colorbar(im_amp, ax=axes[row, 1], fraction=0.046)
        cb_amp.set_label("|Energy|", fontsize=FS_LABEL)
        cb_amp.ax.tick_params(labelsize=FS_TICK)

        for ax in (axes[row, 0], axes[row, 1]):
            ax.add_patch(Rectangle((0.0, -band_kz), band_kx, 2 * band_kz, fill=False,
                                    edgecolor="white", linestyle="--", linewidth=1.3))
            ax.tick_params(labelsize=FS_TICK)

    band_note = ("kz-,kx-band" if domain == "bscan" else
                 "auto-gate band (reference only -- this row's pick is manual)")
    axes[0, 0].legend(handles=[
        Line2D([], [], color="white", linestyle="--", linewidth=1.3, label=band_note),
        Line2D([], [], color="lime", linewidth=1, label="fit population"),
    ], loc="upper right", fontsize=FS_LEGEND, framealpha=0.6)

    color_wls, color_r = "tab:blue", "tab:orange"
    plot_phase_slice(axes[0, 2], KXp, KZp, PHIp, Wp, dx_wls, dz_wls, phi0_wls,
                      xlabel="kx [rad/m]", color=color_wls, line_label="WLS")
    plot_phase_slice(axes[0, 3], KZp, KXp, PHIp, Wp, dz_wls, dx_wls, phi0_wls,
                      xlabel="kz [rad/m]", color=color_wls, line_label="WLS")
    plot_phase_slice(axes[1, 2], KXp, KZp, PHIp, Wp, dx_r, dz_r, phi0_r,
                      xlabel="kx [rad/m]", color=color_r, outlier=outlier_r, line_label="RANSAC")
    plot_phase_slice(axes[1, 3], KZp, KXp, PHIp, Wp, dz_r, dx_r, phi0_r,
                      xlabel="kz [rad/m]", color=color_r, outlier=outlier_r, line_label="RANSAC")
    axes[0, 2].set_title("x-direction fit", fontsize=FS_TITLE)
    axes[0, 3].set_title("z-direction fit", fontsize=FS_TITLE)
    axes[1, 2].set_title("x-direction fit", fontsize=FS_TITLE)
    axes[1, 3].set_title("z-direction fit", fontsize=FS_TITLE)
    for ax in (axes[0, 2], axes[0, 3], axes[1, 2], axes[1, 3]):
        ax.set_ylabel("Phase difference [rad]", fontsize=FS_LABEL)
        ax.xaxis.label.set_fontsize(FS_LABEL)
        ax.tick_params(labelsize=FS_TICK)

    fig.suptitle(f"{method}, {stage_name} ({run_a}→{run_b}), {domain} picking "
                 f"(amp_thr={disp_amp_thr:g}, residual_thr={residual_thr:g})",
                 fontsize=FS_SUPTITLE, fontweight="bold")
    fig.tight_layout()
    return fig, dict(dz=_physical_dz(method, dz_wls), dx=dx_wls, n=n_wls), \
                dict(dz=_physical_dz(method, dz_r), dx=dx_r, n=n_r)


fig, wls_fit, ransac_fit = plot_phase_amp_fit_grid_tuned(
    "gazdag", 3, 8, "bscan", "Chasing", amp_thr=0.35, residual_thr=0.15)

save_fig(fig, "ransac_vs_wls_phase_amp_fit_gazdag_chasing_3_to_8",
         study=STUDY, prefix="FD_", category="Compilations")


In [ ]:
# def sweep_bscan_fit_params(method, run_a, run_b, *,
#                             amp_thr_grid=(0.05, 0.10, 0.15, 0.20, 0.30),
#                             wls_pow_grid=(1, 3, 6),
#                             residual_thr_grid=(0.15, 0.25, 0.35, 0.50),
#                             min_n_mask=15, n_iter=2000, random_state=0):
#     """Grid-search amp_thr x wls_pow x residual_thr for one (method, pair),
#     domain='bscan' only -- kspace picks have no automatic gate to tune (the
#     napari pick IS the fit population); kz_band_fac/kx_band_fac are held at
#     WLS_FIT_DEFAULTS_BY_METHOD[method] (a geometry choice, not a per-pair
#     noise one). Scores each combination by the WLS weighted circular-phase
#     residual (wls_rms, lower = tighter planar fit) and RANSAC's inlier
#     fraction (inlier_frac, higher = more of the gated population agrees on
#     one plane); drops any amp_thr that leaves fewer than min_n_mask cells so
#     an overly strict gate cannot "win" by shrinking to a handful of
#     suspiciously-perfect points. Returns a long-form DataFrame, one row per
#     (amp_thr, wls_pow, residual_thr)."""
#     defaults = WLS_FIT_DEFAULTS_BY_METHOD[method]
#     base_full = post_imaged[method][run_a]
#     mon_full = post_imaged[method][run_b]
#     dz, dx, kz_cent = DZ_BY_METHOD[method], DX_BY_METHOD[method], KZ_CENT_BY_METHOD[method]
#     crop = bscan_masks[(method, run_a, run_b)]
#     z0, z1, x0, x1 = crop["z0"], crop["z1"], crop["x0"], crop["x1"]
#     base = base_full[z0:z1, x0:x1] * crop["soft_win"]
#     mon = mon_full[z0:z1, x0:x1] * crop["soft_win"]

#     records = []
#     for amp_thr in amp_thr_grid:
#         kx_mask = _auto_kx_mask(base, mon, dz, dx, kz_cent,
#                                  defaults["kz_band_fac"], defaults["kx_band_fac"],
#                                  amp_thr, taper="none")
#         n_mask = int(kx_mask.sum())
#         if n_mask < min_n_mask:
#             continue
#         for wls_pow in wls_pow_grid:
#             dz_w, dx_w, phi0_w, n_w, diag_w = wls_phase_plane_fit(
#                 base, mon, dz, dx, kz_cent, taper="none", mask=kx_mask,
#                 wls_pow=wls_pow, return_diagnostics=True)
#             resid = diag_w["phi"][kx_mask] - diag_w["fitted"][kx_mask]
#             resid = (resid + np.pi) % (2 * np.pi) - np.pi  # wrap to (-pi, pi], matches ransac_phase_plane_fit's _wrap_to_pi
#             w = diag_w["w"][kx_mask] ** wls_pow
#             wls_rms = float(np.sqrt(np.sum(w * resid ** 2) / np.sum(w)))
#             for residual_thr in residual_thr_grid:
#                 dz_r, dx_r, phi0_r, n_r = ransac_phase_plane_fit(
#                     base, mon, dz, dx, kz_cent, taper="none", mask=kx_mask,
#                     final_wls_pow=wls_pow, residual_thr=residual_thr,
#                     n_iter=n_iter, random_state=random_state)
#                 records.append(dict(
#                     method=method, pair=(run_a, run_b), amp_thr=amp_thr,
#                     wls_pow=wls_pow, residual_thr=residual_thr, n_mask=n_mask,
#                     dz_wls=_physical_dz(method, dz_w), dx_wls=dx_w, wls_rms=wls_rms,
#                     dz_ransac=_physical_dz(method, dz_r), dx_ransac=dx_r,
#                     n_inliers=n_r, inlier_frac=(n_r / n_mask if n_mask else 0.0)))
#     return pd.DataFrame(records)


# def recommend_bscan_fit_params(sweep_df):
#     """Pick the best row from sweep_bscan_fit_params's output: highest
#     inlier_frac (ties broken by lowest wls_rms). Also reports
#     amp_stability_dz -- the spread (max-min) of dz_ransac across every
#     amp_thr at the winning (wls_pow, residual_thr), a plateau-vs-sharp-
#     optimum indicator: a small spread means the recommendation is not
#     fragile to the exact amp_thr chosen."""
#     if sweep_df.empty:
#         raise ValueError("sweep_df is empty -- every amp_thr left fewer than "
#                           "min_n_mask cells; widen amp_thr_grid or lower min_n_mask.")
#     best = sweep_df.sort_values(["inlier_frac", "wls_rms"], ascending=[False, True]).iloc[0]
#     same_setting = sweep_df[(sweep_df["wls_pow"] == best["wls_pow"]) &
#                              (sweep_df["residual_thr"] == best["residual_thr"])]
#     amp_stability_dz = float(same_setting["dz_ransac"].max() - same_setting["dz_ransac"].min())
#     return dict(method=best["method"], pair=best["pair"], amp_thr=float(best["amp_thr"]),
#                 wls_pow=float(best["wls_pow"]), residual_thr=float(best["residual_thr"]),
#                 inlier_frac=float(best["inlier_frac"]), wls_rms=float(best["wls_rms"]),
#                 dz_wls=float(best["dz_wls"]), dx_wls=float(best["dx_wls"]),
#                 dz_ransac=float(best["dz_ransac"]), dx_ransac=float(best["dx_ransac"]),
#                 amp_stability_dz=amp_stability_dz)


# def plot_bscan_fit_sweep(sweep_df, method, pair, residual_thr_for_dz_panel=None):
#     """Two-panel diagnostic for one (method, pair)'s sweep_bscan_fit_params
#     output. Left: inlier_frac heatmap over (wls_pow, amp_thr), averaged over
#     residual_thr (RANSAC's consensus set barely depends on residual_thr once
#     it is in a sane range, so collapsing it here keeps the heatmap
#     readable). Right: dz_ransac vs amp_thr, one line per wls_pow, at
#     residual_thr_for_dz_panel (default: the sweep's median residual_thr) --
#     a flat plateau means the recommendation is trustworthy, a sharp spike
#     means it is fragile to the exact threshold and worth a closer look."""
#     if residual_thr_for_dz_panel is None:
#         residual_thr_for_dz_panel = float(np.median(sweep_df["residual_thr"].unique()))

#     fig, (ax_heat, ax_dz) = plt.subplots(1, 2, figsize=(12, 5))

#     heat = sweep_df.pivot_table(index="wls_pow", columns="amp_thr", values="inlier_frac", aggfunc="mean")
#     im = ax_heat.imshow(heat.values, aspect="auto", cmap="viridis", vmin=0, vmax=1,
#                          extent=[min(heat.columns), max(heat.columns), max(heat.index), min(heat.index)])
#     ax_heat.set_xticks(list(heat.columns)); ax_heat.set_yticks(list(heat.index))
#     ax_heat.set_xlabel("amp_thr"); ax_heat.set_ylabel("wls_pow")
#     ax_heat.set_title("inlier_frac (mean over residual_thr)")
#     fig.colorbar(im, ax=ax_heat, label="inlier_frac")

#     dz_slice = sweep_df[np.isclose(sweep_df["residual_thr"], residual_thr_for_dz_panel)]
#     for wp in sorted(dz_slice["wls_pow"].unique()):
#         s = dz_slice[dz_slice["wls_pow"] == wp].sort_values("amp_thr")
#         ax_dz.plot(s["amp_thr"], s["dz_ransac"], marker="o", label=f"wls_pow={wp}")
#     ax_dz.axhline(0, color="#888888", lw=0.7, ls="--")
#     ax_dz.set_xlabel("amp_thr"); ax_dz.set_ylabel(r"$\Delta z$ (RANSAC) [m]")
#     ax_dz.set_title(f"residual_thr={residual_thr_for_dz_panel:.2f}")
#     ax_dz.legend(fontsize=8)

#     fig.suptitle(f"{method} {pair[0]}->{pair[1]} B-scan fit-parameter sweep")
#     fig.tight_layout()
#     return fig


# # Re-run with a different METHOD_TO_TUNE to sweep the other two techniques --
# # start with kirchhoff_bp since it is one of the two methods (with backprop)
# # whose Pull-stage B-scan pick showed the widest WLS/RANSAC divergence in the
# # 7.5 summary (see ransac_vs_wls_summary.csv / the Delta_z "erroneous... Pull"
# # note): amp_thr=0.15 (WLS_FIT_DEFAULTS_BY_METHOD's default) let the automatic
# # gate through wide, multi-fringe-direction b-scan regions with no coherent
# # single-plane phase ramp for that pair -- this sweep is meant to find a
# # tighter, more defensible amp_thr/wls_pow/residual_thr instead of guessing.
# METHOD_TO_TUNE = "kirchhoff_bp"

# _sweep_frames = {}
# _recommendations = []
# for _run_a, _run_b in STAGE_PAIRS:
#     _df_sweep = sweep_bscan_fit_params(METHOD_TO_TUNE, _run_a, _run_b)
#     _sweep_frames[(_run_a, _run_b)] = _df_sweep
#     _rec = recommend_bscan_fit_params(_df_sweep)
#     _recommendations.append(_rec)
#     plot_bscan_fit_sweep(_df_sweep, METHOD_TO_TUNE, (_run_a, _run_b))

# pd.DataFrame(_recommendations)

In [ ]:
records = []
for method in ("kirchhoff_bp", "gazdag", "backprop"):
    for (run_a, run_b), stage_name in zip(STAGE_PAIRS, STAGE_NAMES):
        for domain in ("kspace", "bscan"):
            # _fit_inputs always supplies an explicit mask= for both domains
            # (see its docstring / restricting_kx.md), so no pos_kz_only/
            # pos_kx_only override is needed here -- mask=... bypasses that
            # logic entirely regardless of its value.
            base, mon, dz, dx, kz_cent, kwargs, wls_pow = _fit_inputs(method, run_a, run_b, domain)
            dz_wls, dx_wls, phi0_wls, n_wls = wls_phase_plane_fit(
                base, mon, dz, dx, kz_cent, wls_pow=6, amp_thr=0.15, **kwargs)
            dz_r, dx_r, phi0_r, n_r = ransac_phase_plane_fit(
                base, mon, dz, dx, kz_cent, final_wls_pow=6, n_iter=2000,
                residual_thr=0.15, random_state=0, amp_thr=0.15, **kwargs)
            # _physical_dz: back-propagation's dz needs the sign flip described by
            # DZ_SIGN_BY_METHOD above -- without it, this table (and  the grouped bar
            # chart built from it) shows backprop's Delta_z pointing the opposite
            # physical direction from Kirchhoff-BP/Gazdag's, for every stage.
            records.append(dict(method=method, stage=stage_name, pair=(run_a, run_b),
                                 domain=domain, dz_wls=_physical_dz(method, dz_wls), dx_wls=dx_wls,
                                 dz_ransac=_physical_dz(method, dz_r), dx_ransac=dx_r))

ransac_vs_wls = pd.DataFrame(records)
ransac_vs_wls.to_csv(OUT_DIR / "roi_phase" / "ransac_vs_wls_summary.csv", index=False)

In [ ]:
from matplotlib.patches import Patch

METHOD_LABELS = {"gazdag": "Gazdag", "kirchhoff_bp": "Kirchhoff-BP", "backprop": "Back-propagation"}
METHOD_COLORS = {"gazdag": "#0072B2", "kirchhoff_bp": "#E69F00", "backprop": "#009E73"}


def plot_ransac_vs_wls_summary(df, domain):
    """One panel-pair of Figure 7.5-2 (call once per domain: kspace, bscan) --
    styled to match FieldData_Playground.ipynb's summary bar chart (old cells
    [106]/[114]): Delta_z and Delta_x per stage, all three methods' bars
    grouped together within each stage (not laid out flat one-bar-per-row),
    each method its own colour, hatched=WLS vs solid=RANSAC. The chapter
    combines both domains' figures into a single #figure() via
    subfigs(cols: 1, ...)."""
    sub = df[df["domain"] == domain]
    stages = [s for s in STAGE_NAMES if s in sub["stage"].values]
    methods = [m for m in ("kirchhoff_bp", "gazdag", "backprop") if m in sub["method"].values]

    n_method = len(methods)
    group_w = 0.8
    bar_w = group_w / (n_method * 2)
    x = np.arange(len(stages))

    fig, (ax_dz, ax_dx) = plt.subplots(1, 2, figsize=(13, 5))
    for mi, method in enumerate(methods):
        m_sub = sub[sub["method"] == method].set_index("stage")
        dz_w = np.array([m_sub["dz_wls"].get(s, np.nan) for s in stages])
        dz_r = np.array([m_sub["dz_ransac"].get(s, np.nan) for s in stages])
        dx_w = np.array([m_sub["dx_wls"].get(s, np.nan) for s in stages])
        dx_r = np.array([m_sub["dx_ransac"].get(s, np.nan) for s in stages])
        pos_w = x - group_w / 2 + (2 * mi) * bar_w + bar_w / 2
        pos_r = pos_w + bar_w
        c = METHOD_COLORS[method]
        ax_dz.bar(pos_w, dz_w, width=bar_w, color=c, hatch="///", edgecolor="white", linewidth=0.6)
        ax_dz.bar(pos_r, dz_r, width=bar_w, color=c, edgecolor="white", linewidth=0.6)
        ax_dx.bar(pos_w, dx_w, width=bar_w, color=c, hatch="///", edgecolor="white", linewidth=0.6)
        ax_dx.bar(pos_r, dx_r, width=bar_w, color=c, edgecolor="white", linewidth=0.6)

    for ax, ylabel, title in ((ax_dz, r"$\Delta z$ (depth) [m]", r"$\Delta z$ per stage"),
                               (ax_dx, r"$\Delta x$ (radial) [m]", r"$\Delta x$ per stage")):
        ax.axhline(0, color="#888888", lw=0.7, ls="--", zorder=0)
        ax.set_xticks(x); ax.set_xticklabels(stages)
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    method_handles = [Patch(facecolor=METHOD_COLORS[m], edgecolor="white", label=METHOD_LABELS[m])
                       for m in methods]
    fit_handles = [Patch(facecolor="#999999", hatch="///", edgecolor="white", label="WLS"),
                   Patch(facecolor="#999999", edgecolor="white", label="RANSAC")]
    leg1 = ax_dx.legend(handles=method_handles, loc="upper left", bbox_to_anchor=(1.02, 1.0),
                         fontsize=9, title="Method", frameon=False)
    ax_dx.add_artist(leg1)
    ax_dx.legend(handles=fit_handles, loc="upper left", bbox_to_anchor=(1.02, 0.55),
                 fontsize=9, title="Fit", frameon=False)

    domain_label = "wavenumber (k-space)" if domain == "kspace" else "amplitude (B-scan)"
    fig.suptitle("WLS vs RANSAC displacement estimate per stage and migration method -- "
                 f"{domain_label} picking (hatched = WLS, solid = RANSAC)", y=1.03)
    fig.tight_layout()
    return fig


fig_kspace = plot_ransac_vs_wls_summary(ransac_vs_wls, "kspace")
save_fig(fig_kspace, "ransac_vs_wls_displacement_summary", study=STUDY, prefix="FD_", category="Compilations")

fig_bscan = plot_ransac_vs_wls_summary(ransac_vs_wls, "bscan")
save_fig(fig_bscan, "ransac_vs_wls_bscan_displacement_summary", study=STUDY, prefix="FD_", category="Compilations")

## Notes for the reader

- Cells producing `FD_borehole_domain_schematic.png` and `FD_ransac_vs_wls_*_displacement_summary.png`
  are the closest to drop-in ports of the old notebook and should need the least rework.
- `apply_post_imaging`, `launch_napari_picker`/`harvest_napari_mask`, and
  `stage_anchored_trajectory` are the three genuinely new pieces of shared
  logic this rewrite introduces; consider promoting them into
  `helper_functions/migration.py` once validated here, alongside
  `wls_phase_plane_fit`/`ransac_phase_plane_fit`.
- *Optional, non-blocking simplification*: `ransac_phase_plane_fit` currently
  duplicates `wls_phase_plane_fit`'s weighted-lstsq block for its own final
  inlier refit rather than calling `wls_phase_plane_fit(mask=inliers,
  wls_pow=final_wls_pow)`. Not changed here since it's out of this rewrite's
  scope, but flagged since it would remove a second copy of that math.